In [ ]:
from pathlib import Path
import os
import sys
_release_root = Path(os.environ.get('AVATAR_RELEASE_ROOT', Path.cwd())).resolve()
while not (_release_root / 'avatar_paths.py').is_file():
    if _release_root == _release_root.parent:
        raise RuntimeError('Open this notebook from the code package directory.')
    _release_root = _release_root.parent
if str(_release_root) not in sys.path:
    sys.path.insert(0, str(_release_root))
from avatar_paths import project_path, get_value, get_column, get_path, get_label


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import math
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import PathPatch, Rectangle
from matplotlib.path import Path as MplPath
from matplotlib.lines import Line2D
plt.rcParams.update({'font.family': 'Arial', 'font.sans-serif': ['Arial'], 'font.size': 11.0, 'axes.titlesize': 13.0, 'axes.labelsize': 11.5, 'xtick.labelsize': 10.5, 'ytick.labelsize': 10.5, 'legend.fontsize': 10.0, 'legend.title_fontsize': 10.5, 'axes.linewidth': 1.05, 'xtick.major.width': 1.0, 'ytick.major.width': 1.0, 'xtick.major.size': 4.0, 'ytick.major.size': 4.0, 'lines.linewidth': 1.35, 'lines.markersize': 7.0, 'patch.linewidth': 1.0, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none'})
MBP_ORDER = ['MBP-I', 'MBP-II', 'MBP-III']
MBP_COLORS = {'MBP-I': '#2F6FA3', 'MBP-II': '#C6862C', 'MBP-III': '#C85A54'}
ENDPOINT_ORDER = ['Response', 'PFS', 'OS']
ENDPOINT_COLORS = {'Response': '#C85A54', 'PFS': '#3A76A8', 'OS': '#7B59A5'}
COLOR_TEXT = '#222222'
COLOR_MUTED = '#555555'
COLOR_GRID = '#D7D7D7'
COLOR_LIGHT = '#B8B8B8'
FIG_DPI = 600
MODALITY_ORDER_STAGE1 = ['Metagenomics', 'Radiomics', 'Blood', 'Urine']
MODALITY_ORDER_MRU = ['Radiomics', 'Metagenomics', 'Urine']
MODALITY_COLORS = {'Radiomics': '#355F9A', 'Metagenomics': '#23806D', 'Microbiota': '#23806D', 'Blood': '#B5762A', 'Urine': '#7B59A5'}
BENEFIT_CMAP = LinearSegmentedColormap.from_list('benefit_diverging', ['#356B97', '#F4F4F4', '#C85A54'], N=256)
FEATURE_VALUE_CMAP = LinearSegmentedColormap.from_list('feature_value_diverging', ['#356B97', '#F4F4F4', '#C85A54'], N=256)



In [ ]:
PROJECT_ROOT = Path(str(project_path('')))
RESULT_ROOTS = [PROJECT_ROOT / get_value('science.paths.directory_008', 'directory_008') / get_value('science.paths.directory_010', 'directory_010'), PROJECT_ROOT / get_value('science.paths.directory_010', 'directory_010')]

def resolve_stage_dir(stage_name, required_filename):
    candidates = [root / stage_name for root in RESULT_ROOTS]
    for d in candidates:
        if (d / required_filename).exists():
            
            return d
    raise FileNotFoundError(f"Cannot locate {stage_name} containing '{required_filename}'.\n" + '\n'.join((f'  - {d}' for d in candidates)))

def resolve_existing_file(relative_parts):
    candidates = [root.joinpath(*relative_parts) for root in RESULT_ROOTS]
    for p in candidates:
        if p.exists():
            
            return p
    print('[Optional file not found]')
    for p in candidates:
        pass
    return candidates[0]
STAGE1_DIR = resolve_stage_dir(get_value('science.paths.directory_003', 'directory_003'), get_value('science.tables.result_005', 'result_005.csv'))
MRU_DIR = resolve_stage_dir(get_value('science.paths.directory_028', 'directory_028'), get_value('science.tables.features_015', 'features_015.csv'))
MBP_PATH = resolve_existing_file([get_value('science.paths.directory_011', 'directory_011'), get_value('science.paths.directory_026', 'directory_026'), get_value('science.tables.result_025', 'result_025.csv')])
MODEL_RESULTS_DIR = MRU_DIR.parent
OUT_DIR = MODEL_RESULTS_DIR / get_value('science.paths.directory_121', 'directory_121')
MAIN_OUT_DIR = OUT_DIR / 'Main'
SOURCE_OUT_DIR = OUT_DIR / get_value('science.paths.directory_122', 'directory_122')
for d in [OUT_DIR, MAIN_OUT_DIR, SOURCE_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
TOP_SHARED = 5
TOP_PREFERRED_PER_ENDPOINT = 3
MAX_PATIENT_HEATMAP_FEATURES = 15
BOOTSTRAP_N = 500
BOOTSTRAP_SEED = 20260810








In [ ]:
def require_file(path, label=None):
    path = Path(path)
    if not path.exists():
        name = label or path.name
        raise FileNotFoundError(f'Required file not found: {name}\n{path}\nPlease check PROJECT_ROOT / MODEL_RESULTS_DIR.')
    return path

def optional_file(path):
    path = Path(path)
    return path if path.exists() else None

def save_figure(fig, stem, save_svg=True):
    stem = Path(stem)
    stem.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(stem) + '.png', dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    fig.savefig(str(stem) + '.pdf', bbox_inches='tight', facecolor='white')
    if save_svg:
        fig.savefig(str(stem) + '.svg', bbox_inches='tight', facecolor='white')
    

def clean_axes(ax, grid_axis=None):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(colors=COLOR_TEXT)
    if grid_axis is not None:
        ax.grid(axis=grid_axis, color=COLOR_GRID, linewidth=0.8, alpha=1.0, zorder=0)

def add_panel_header(fig, letter=None, title=None, x_letter=0.012, x_title=0.06, y=0.988):
    return

def normalize_sample_id(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if re.fullmatch('-?\\d+\\.0', s):
        s = s[:-2]
    return s.lower()

def read_indexed_csv(path):
    path = require_file(path)
    df = pd.read_csv(path, encoding='utf-8-sig')
    first = str(df.columns[0]).strip()
    first_lower = first.lower()
    if first_lower in {'sample', 'sample_id', 'sample id', 'id', 'unnamed: 0'}:
        idx = df.iloc[:, 0].map(normalize_sample_id)
        df = df.iloc[:, 1:].copy()
        df.index = idx
    else:
        converted = pd.to_numeric(df.iloc[:, 0], errors='coerce')
        if converted.isna().mean() > 0.8:
            idx = df.iloc[:, 0].map(normalize_sample_id)
            df = df.iloc[:, 1:].copy()
            df.index = idx
    df.index = [normalize_sample_id(x) for x in df.index]
    return df

def read_table(path):
    path = require_file(path)
    return pd.read_csv(path, encoding='utf-8-sig')

def find_col(df, candidates, required=True):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if str(cand).strip().lower() in lower_map:
            return lower_map[str(cand).strip().lower()]
    if required:
        raise ValueError(f'Cannot find any of {candidates}. Existing columns:\n{df.columns.tolist()}')
    return None

def clean_modality_name(x):
    s = str(x).strip()
    mapper = {'microbiota': 'Metagenomics', 'microbiome': 'Metagenomics', 'metagenomics': 'Metagenomics', 'radiomics': 'Radiomics', 'blood': 'Blood', 'urine': 'Urine'}
    return mapper.get(s.lower(), s)

def clean_outcome_name(x):
    s = str(x).strip()
    mapper = {'response': 'Response', 'resp': 'Response', 'pfs': 'PFS', 'os': 'OS'}
    return mapper.get(s.lower(), s)

def clean_feature_label(x, max_len=42):
    s = str(x)
    s = re.sub('^(Metagenomics|Microbiota|Microbiome|Radiomics|Blood|Urine)[_:]{1,2}', '', s, flags=re.I)
    s = s.replace('_', ' ')
    s = re.sub('\\s+', ' ', s).strip()
    if len(s) > max_len:
        s = s[:max_len - 1] + '…'
    return s

def fmt_n(x):
    x = int(round(float(x)))
    return f'{x:,}'

def rank_spearman(x, y):
    x = pd.Series(x, dtype=float)
    y = pd.Series(y, dtype=float)
    mask = x.notna() & y.notna()
    if mask.sum() < 4:
        return np.nan
    return x[mask].rank(method='average').corr(y[mask].rank(method='average'), method='pearson')

def robust_row_scale(df, quantile=0.95):
    out = df.copy().astype(float)
    for idx in out.index:
        vals = out.loc[idx].values.astype(float)
        denom = np.nanquantile(np.abs(vals), quantile)
        if not np.isfinite(denom) or denom <= 0:
            denom = np.nanmax(np.abs(vals))
        if not np.isfinite(denom) or denom <= 0:
            denom = 1.0
        out.loc[idx] = np.clip(vals / denom, -1, 1)
    return out

def parse_selected_outcomes(x):
    if pd.isna(x):
        return []
    parts = re.split('[;,|+/]+', str(x))
    out = []
    for p in parts:
        p = clean_outcome_name(p.strip())
        if p in ENDPOINT_ORDER and p not in out:
            out.append(p)
    return out

def endpoint_pattern(outcomes):
    s = set(outcomes)
    if s == {'Response'}:
        return 'Response-specific'
    if s == {'PFS'}:
        return 'PFS-specific'
    if s == {'OS'}:
        return 'OS-specific'
    if len(s) == 2:
        return 'Shared by 2'
    if len(s) >= 3:
        return 'Shared by 3'
    return 'Other'
PATTERN_ORDER = ['Response-specific', 'PFS-specific', 'OS-specific', 'Shared by 2', 'Shared by 3']


In [ ]:
STAGE1_PERF_PATH = STAGE1_DIR / get_value('science.tables.metrics_002', 'metrics_002.csv')
STAGE1_GOLDEN_PATH = STAGE1_DIR / get_value('science.tables.result_005', 'result_005.csv')
STAGE1_TOP_PATH = STAGE1_DIR / get_value('science.tables.result_004', 'result_004.csv')
stage1_perf = read_table(STAGE1_PERF_PATH)
golden = read_table(STAGE1_GOLDEN_PATH)
stage1_top = read_table(STAGE1_TOP_PATH)
perf_modal_col = find_col(stage1_perf, ['modal', 'modality'])
perf_nfeat_col = find_col(stage1_perf, ['n_features', 'n_feature', 'features'])
gold_modal_col = find_col(golden, ['modal', 'modality'])
gold_feat_col = find_col(golden, ['feature'])
gold_outcomes_col = find_col(golden, ['selected_outcomes'])
gold_nout_col = find_col(golden, ['n_outcomes'], required=False)
stage1_perf['modal_clean'] = stage1_perf[perf_modal_col].map(clean_modality_name)
golden['modal_clean'] = golden[gold_modal_col].map(clean_modality_name)
golden['feature_clean'] = golden[gold_feat_col].astype(str)
golden['outcome_list'] = golden[gold_outcomes_col].map(parse_selected_outcomes)
golden['pattern'] = golden['outcome_list'].map(endpoint_pattern)
print('Stage 1 performance:', stage1_perf.shape)
print('Golden features:', golden.shape)
display(golden.groupby('modal_clean')['feature_clean'].nunique().rename('retained_features').to_frame())


In [ ]:
def plot_fig4_feature_compression():
    raw_counts = stage1_perf[['modal_clean', perf_nfeat_col]].drop_duplicates('modal_clean').rename(columns={perf_nfeat_col: 'input_features'})
    retained_counts = golden.groupby('modal_clean')['feature_clean'].nunique().rename('retained_features').reset_index()
    d = raw_counts.merge(retained_counts, on='modal_clean', how='inner')
    order = [m for m in MODALITY_ORDER_STAGE1 if m in d['modal_clean'].values]
    extras = [m for m in d['modal_clean'] if m not in order]
    order += extras
    d['modal_clean'] = pd.Categorical(d['modal_clean'], categories=order, ordered=True)
    d = d.sort_values('modal_clean').reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(6.5, 3.3), facecolor='white')
    y = np.arange(len(d))
    for i, r in d.iterrows():
        modal = str(r['modal_clean'])
        raw = float(r['input_features'])
        kept = float(r['retained_features'])
        c = MODALITY_COLORS.get(modal, '#777777')
        ax.plot([kept, raw], [i, i], color='#B8B8B8', linewidth=1.3, zorder=1)
        ax.scatter(raw, i, s=67, facecolor='white', edgecolor='#777777', linewidth=1.0, zorder=3)
        ax.scatter(kept, i, s=81, facecolor=c, edgecolor='white', linewidth=0.8, zorder=4)
        ax.text(raw * 1.08, i, fmt_n(raw), ha='left', va='center', fontsize=10.0, color=COLOR_MUTED)
        ax.text(kept / 1.18, i, fmt_n(kept), ha='right', va='center', fontsize=10.0, fontweight='bold', color=c)
    ax.set_xscale('log')
    ax.set_yticks(y)
    ax.set_yticklabels(d['modal_clean'].astype(str))
    ax.set_xlabel('Number of features (log scale)')
    ax.set_ylabel('')
    ax.set_ylim(len(d) - 0.5, -0.5)
    clean_axes(ax, grid_axis='x')
    handles = [Line2D([0], [0], marker='o', linestyle='', markerfacecolor='white', markeredgecolor='#777777', markersize=7.0, label='Input features'), Line2D([0], [0], marker='o', linestyle='', markerfacecolor=COLOR_TEXT, markeredgecolor='white', markersize=7.5, label='SHAP-retained')]
    ax.legend(handles=handles, loc='lower right', frameon=False, fontsize=10.0, ncol=2)
    add_panel_header(fig, '4D', 'SHAP-guided compression of high-dimensional modality-specific inputs')
    fig.subplots_adjust(left=0.2, right=0.88, top=0.82, bottom=0.13)
    source = d.copy()
    source.to_csv(SOURCE_OUT_DIR / get_value('science.tables.attribution_028', 'attribution_028.csv'), index=False, encoding='utf-8-sig')
    save_figure(fig, MAIN_OUT_DIR / get_value('science.paths.directory_123', 'directory_123'))
    plt.show()
    return source


In [ ]:
def plot_fig4_endpoint_overlap():
    counts = golden.groupby(['modal_clean', 'pattern'])['feature_clean'].nunique().rename('n_features').reset_index()
    order = [m for m in MODALITY_ORDER_STAGE1 if m in counts['modal_clean'].unique()]
    extras = [m for m in counts['modal_clean'].unique() if m not in order]
    order += extras
    grid = pd.MultiIndex.from_product([order, PATTERN_ORDER], names=['modal_clean', 'pattern']).to_frame(index=False).merge(counts, on=['modal_clean', 'pattern'], how='left')
    grid['n_features'] = grid['n_features'].fillna(0).astype(int)
    max_n = max(int(grid['n_features'].max()), 1)
    fig, ax = plt.subplots(figsize=(7.5, 3.5), facecolor='white')
    for i, modal in enumerate(order):
        for j, pattern in enumerate(PATTERN_ORDER):
            n = int(grid.loc[(grid['modal_clean'] == modal) & (grid['pattern'] == pattern), 'n_features'].iloc[0])
            if n <= 0:
                ax.scatter(j, i, s=28, facecolor='white', edgecolor='#D8D8D8', linewidth=0.7, zorder=2)
                continue
            size = 45 + 520 * (n / max_n) ** 0.85
            c = MODALITY_COLORS.get(modal, '#777777')
            ax.scatter(j, i, s=size, facecolor=c, edgecolor='white', linewidth=0.9, alpha=0.98, zorder=3)
            text_color = 'white' if size >= 110 else COLOR_TEXT
            ax.text(j, i, str(n), ha='center', va='center', fontsize=9.7, fontweight='bold', color=text_color, zorder=4)
    ax.set_xticks(np.arange(len(PATTERN_ORDER)))
    ax.set_xticklabels(['Response\nspecific', 'PFS\nspecific', 'OS\nspecific', 'Shared by\n2 endpoints', 'Shared by\n3 endpoints'])
    ax.set_yticks(np.arange(len(order)))
    ax.set_yticklabels(order)
    ax.set_xlim(-0.6, len(PATTERN_ORDER) - 0.4)
    ax.set_ylim(len(order) - 0.5, -0.5)
    ax.grid(color=COLOR_GRID, linewidth=0.55, zorder=0)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    add_panel_header(fig, '4E', 'Endpoint overlap among SHAP-retained features')
    fig.subplots_adjust(left=0.18, right=0.98, top=0.82, bottom=0.14)
    grid.to_csv(SOURCE_OUT_DIR / get_value('science.tables.attribution_029', 'attribution_029.csv'), index=False, encoding='utf-8-sig')
    save_figure(fig, MAIN_OUT_DIR / get_value('science.paths.directory_124', 'directory_124'))
    plt.show()
    return grid


In [ ]:
def _stage1_shap_abs_path(modal, task):
    return STAGE1_DIR / modal / get_value('science.paths.directory_004', 'directory_004') / f'{modal}_cv_shap_abs_{task}.csv.gz'

def _infer_topk_by_modal_endpoint():
    modal_col = find_col(stage1_top, ['modal', 'modality'])
    outcome_col = find_col(stage1_top, ['outcome', 'task', 'endpoint'])
    feature_col = find_col(stage1_top, ['feature'])
    tmp = stage1_top.copy()
    tmp['modal_clean'] = tmp[modal_col].map(clean_modality_name)
    tmp['outcome_clean'] = tmp[outcome_col].map(clean_outcome_name)
    return tmp.groupby(['modal_clean', 'outcome_clean'])[feature_col].nunique().to_dict()

def build_stage1_bootstrap_stability(n_bootstrap=BOOTSTRAP_N, random_state=BOOTSTRAP_SEED):
    rng = np.random.default_rng(random_state)
    topk_map = _infer_topk_by_modal_endpoint()
    endpoint_records = []
    for modal in MODALITY_ORDER_STAGE1:
        gf = golden[golden['modal_clean'] == modal].copy()
        if gf.empty:
            continue
        retained_features = set(gf['feature_clean'].astype(str))
        for task in ENDPOINT_ORDER:
            p = _stage1_shap_abs_path(modal, task)
            if not p.exists():
                print(f'[Skip] missing Stage1 OOF SHAP: {p}')
                continue
            S = read_indexed_csv(p)
            S = S.apply(pd.to_numeric, errors='coerce')
            retained_available = [f for f in retained_features if f in S.columns]
            if not retained_available:
                continue
            top_k = int(topk_map.get((modal, task), 1))
            top_k = max(1, min(top_k, S.shape[1]))
            n = S.shape[0]
            freq = {f: 0 for f in retained_available}
            ranks = {f: [] for f in retained_available}
            arr = S.values.astype(float)
            cols = np.asarray(S.columns, dtype=object)
            for _ in range(n_bootstrap):
                idx = rng.integers(0, n, size=n)
                imp = np.nanmean(arr[idx, :], axis=0)
                order = np.argsort(-np.nan_to_num(imp, nan=-np.inf), kind='mergesort')
                rank_arr = np.empty(len(order), dtype=int)
                rank_arr[order] = np.arange(1, len(order) + 1)
                top_features = set(cols[order[:top_k]])
                for f in retained_available:
                    j = S.columns.get_loc(f)
                    ranks[f].append(int(rank_arr[j]))
                    if f in top_features:
                        freq[f] += 1
            selected_by_original = set(stage1_top.loc[(stage1_top[find_col(stage1_top, ['modal', 'modality'])].map(clean_modality_name) == modal) & (stage1_top[find_col(stage1_top, ['outcome', 'task', 'endpoint'])].map(clean_outcome_name) == task), find_col(stage1_top, ['feature'])].astype(str))
            for f in retained_available:
                rr = np.asarray(ranks[f], dtype=float)
                endpoint_records.append({'modal_clean': modal, 'feature_clean': f, 'outcome_clean': task, 'originally_selected': int(f in selected_by_original), 'top_k': top_k, 'bootstrap_topk_frequency': freq[f] / n_bootstrap, 'bootstrap_median_rank': float(np.nanmedian(rr)), 'bootstrap_rank_q25': float(np.nanquantile(rr, 0.25)), 'bootstrap_rank_q75': float(np.nanquantile(rr, 0.75)), 'n_bootstrap': n_bootstrap})
    endpoint_df = pd.DataFrame(endpoint_records)
    selected = endpoint_df[endpoint_df['originally_selected'] == 1].copy()
    feature_df = selected.groupby(['modal_clean', 'feature_clean'], as_index=False).agg(mean_topk_frequency=('bootstrap_topk_frequency', 'mean'), min_topk_frequency=('bootstrap_topk_frequency', 'min'), median_bootstrap_rank=('bootstrap_median_rank', 'median'), n_selected_endpoints=('outcome_clean', 'nunique'))
    feature_df = feature_df.merge(golden[['modal_clean', 'feature_clean', 'pattern']].drop_duplicates(), on=['modal_clean', 'feature_clean'], how='left')
    endpoint_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_202', 'result_202.csv'), index=False, encoding='utf-8-sig')
    feature_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_203', 'result_203.csv'), index=False, encoding='utf-8-sig')
    return (endpoint_df, feature_df)
stage1_stability_endpoint, stage1_stability_feature = build_stage1_bootstrap_stability()

def plot_fig4f_bootstrap_stability():
    d = stage1_stability_feature.copy()
    order = [m for m in MODALITY_ORDER_STAGE1 if m in d['modal_clean'].unique()]
    y_base = {m: i for i, m in enumerate(order)}
    fig, ax = plt.subplots(figsize=(7.8, 3.5), facecolor='white')
    rng = np.random.default_rng(42)
    for modal in order:
        ss = d[d['modal_clean'] == modal].copy()
        if ss.empty:
            continue
        jitter = np.linspace(-0.17, 0.17, len(ss)) if len(ss) > 1 else np.array([0.0])
        ss = ss.sort_values('mean_topk_frequency').reset_index(drop=True)
        y = y_base[modal] + jitter
        sizes = 26 + 24 * ss['n_selected_endpoints'].clip(1, 3).values
        c = MODALITY_COLORS.get(modal, '#777777')
        ax.scatter(ss['mean_topk_frequency'], y, s=sizes, facecolor=c, edgecolor='white', linewidth=0.65, alpha=0.98, zorder=3)
        med = float(ss['mean_topk_frequency'].median())
        q25 = float(ss['mean_topk_frequency'].quantile(0.25))
        q75 = float(ss['mean_topk_frequency'].quantile(0.75))
        ax.plot([q25, q75], [y_base[modal], y_base[modal]], color=COLOR_TEXT, linewidth=2.1, solid_capstyle='round', zorder=4)
        ax.scatter(med, y_base[modal], s=48, facecolor='white', edgecolor=COLOR_TEXT, linewidth=1.0, zorder=5)
    ax.axvline(0.5, color='#AAAAAA', linewidth=0.8, linestyle=(0, (3, 3)), zorder=1)
    ax.set_yticks(np.arange(len(order)))
    ax.set_yticklabels(order)
    ax.set_xlim(-0.02, 1.02)
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.set_xticklabels([f'{int(x * 100)}%' for x in np.linspace(0, 1, 6)])
    ax.set_xlabel('Bootstrap Top-k retention frequency')
    ax.set_ylabel('')
    ax.set_ylim(len(order) - 0.5, -0.5)
    clean_axes(ax, grid_axis='x')
    handles = [ax.scatter([], [], s=26 + 24 * k, facecolor='#777777', edgecolor='white', linewidth=0.6) for k in [1, 2, 3]]
    ax.legend(handles, ['1 endpoint', '2 endpoints', '3 endpoints'], title='Originally selected in', loc='center left', bbox_to_anchor=(1.015, 0.5), frameon=False, fontsize=9.7, title_fontsize=9.7, borderaxespad=0.0)
    add_panel_header(fig, '4F', 'Bootstrap robustness of Stage-1 SHAP feature rankings')
    fig.subplots_adjust(left=0.17, right=0.76, top=0.82, bottom=0.14)
    save_figure(fig, MAIN_OUT_DIR / get_value('science.paths.directory_125', 'directory_125'))
    plt.show()
    return d


In [ ]:
MRU_X_PATH = MRU_DIR / get_value('science.tables.features_015', 'features_015.csv')
MRU_PRED_PATH = MRU_DIR / get_value('science.tables.predictions_016', 'predictions_016.csv')
MRU_IMP_PATH = MRU_DIR / get_value('science.tables.attribution_013', 'attribution_013.csv')
MRU_META_PATH = MRU_DIR / get_value('science.tables.result_028', 'result_028.csv')
MRU_MODAL_CONTRIB_PATH = MRU_DIR / get_value('science.tables.attribution_014', 'attribution_014.csv')
X_mru = read_indexed_csv(MRU_X_PATH)
pred_mru = read_indexed_csv(MRU_PRED_PATH)
imp_mru = read_table(MRU_IMP_PATH)
meta_mru = read_table(MRU_META_PATH)
modal_contrib_mru = read_table(MRU_MODAL_CONTRIB_PATH)
shap_signed_raw = {}
shap_abs_mru = {}
for task in ENDPOINT_ORDER:
    shap_signed_raw[task] = read_indexed_csv(MRU_DIR / f'MRU_final_shap_signed_{task}.csv.gz')
    shap_abs_mru[task] = read_indexed_csv(MRU_DIR / f'MRU_final_shap_abs_{task}.csv.gz')
shap_benefit = {}
for task in ENDPOINT_ORDER:
    shap_benefit[task] = shap_signed_raw[task].copy()
    if task in {'PFS', 'OS'}:
        shap_benefit[task] = -shap_benefit[task]
imp_outcome_col = find_col(imp_mru, ['outcome', 'task', 'endpoint'])
imp_feature_col = find_col(imp_mru, ['feature'])
imp_abs_col = find_col(imp_mru, ['mean_abs_shap', 'mean_shap', 'shap_importance'])
imp_mru['outcome_clean'] = imp_mru[imp_outcome_col].map(clean_outcome_name)
imp_mru['feature_clean'] = imp_mru[imp_feature_col].astype(str)
imp_mru['mean_abs_shap_clean'] = pd.to_numeric(imp_mru[imp_abs_col], errors='coerce')
meta_feature_col = find_col(meta_mru, ['feature', 'selected_feature', 'feature_name'])
meta_modal_col = find_col(meta_mru, ['modal', 'modality', 'feature_type'])
meta_lookup = meta_mru[[meta_feature_col, meta_modal_col]].drop_duplicates(meta_feature_col).rename(columns={meta_feature_col: 'feature_clean', meta_modal_col: 'modal_clean'})
meta_lookup['feature_clean'] = meta_lookup['feature_clean'].astype(str)
meta_lookup['modal_clean'] = meta_lookup['modal_clean'].map(clean_modality_name)
imp_mru = imp_mru.merge(meta_lookup, on='feature_clean', how='left')

def infer_modal_from_feature(feature):
    s = str(feature).lower()
    if s.startswith('radiomics'):
        return 'Radiomics'
    if s.startswith(('metagenomics', 'microbiota', 'microbiome', 'ko')):
        return 'Metagenomics'
    if s.startswith('urine'):
        return 'Urine'
    if s.startswith('blood'):
        return 'Blood'
    return np.nan
imp_mru['modal_clean'] = imp_mru['modal_clean'].fillna(imp_mru['feature_clean'].map(infer_modal_from_feature))
print('MRU X:', X_mru.shape)
print('MRU predictions:', pred_mru.shape)
print('MRU SHAP importance:', imp_mru.shape)
print('MRU metadata:', meta_mru.shape)
display(imp_mru.groupby(['outcome_clean', 'modal_clean'])['mean_abs_shap_clean'].sum().unstack(fill_value=0))


In [ ]:
def build_all_feature_preference_table():
    d = imp_mru[imp_mru['outcome_clean'].isin(ENDPOINT_ORDER)].copy()
    denom = d.groupby('outcome_clean')['mean_abs_shap_clean'].transform('sum')
    d['relative_importance'] = d['mean_abs_shap_clean'] / denom.replace(0, np.nan)
    wide = d.pivot_table(index=['feature_clean', 'modal_clean'], columns='outcome_clean', values='relative_importance', aggfunc='mean', fill_value=0.0).reset_index()
    for task in ENDPOINT_ORDER:
        if task not in wide.columns:
            wide[task] = 0.0
    wide['overall_relative_importance'] = wide[ENDPOINT_ORDER].mean(axis=1)
    wide['sum_relative_importance'] = wide[ENDPOINT_ORDER].sum(axis=1)
    total = wide['sum_relative_importance'].replace(0, np.nan)
    for task in ENDPOINT_ORDER:
        wide[f'p_{task}'] = wide[task] / total
    p = wide[[f'p_{t}' for t in ENDPOINT_ORDER]].fillna(1 / 3).values.astype(float)
    eps = 1e-12
    entropy = -(p * np.log(p + eps)).sum(axis=1) / np.log(3.0)
    wide['sharedness_entropy'] = entropy
    p_sorted = np.sort(p, axis=1)
    wide['preference_margin'] = p_sorted[:, -1] - p_sorted[:, -2]
    wide['dominant_endpoint'] = [ENDPOINT_ORDER[int(i)] for i in np.argmax(p, axis=1)]
    wide['shared_score'] = wide['overall_relative_importance'] * wide['sharedness_entropy']
    for task in ENDPOINT_ORDER:
        others = [x for x in ENDPOINT_ORDER if x != task]
        wide[f'{task}_preference_score'] = wide['overall_relative_importance'] * np.maximum(wide[f'p_{task}'] - 0.5 * (wide[f'p_{others[0]}'] + wide[f'p_{others[1]}']), 0)
    direction_records = []
    for _, r in d.iterrows():
        task = r['outcome_clean']
        feature = r['feature_clean']
        if feature in X_mru.columns and feature in shap_benefit[task].columns:
            common = X_mru.index.intersection(shap_benefit[task].index)
            rho = rank_spearman(X_mru.loc[common, feature], shap_benefit[task].loc[common, feature])
        else:
            rho = np.nan
        direction_records.append(rho)
    d['direction_rho'] = direction_records
    wide.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_204', 'result_204.csv'), index=False, encoding='utf-8-sig')
    d.to_csv(SOURCE_OUT_DIR / get_value('science.tables.attribution_030', 'attribution_030.csv'), index=False, encoding='utf-8-sig')
    return (d, wide)
determinant_long, feature_preference = build_all_feature_preference_table()
display(feature_preference.sort_values('overall_relative_importance', ascending=False).head(12))


In [ ]:
def plot_fig6a_modality_attribution():
    d = modal_contrib_mru.copy()
    modal_col = find_col(d, ['modal', 'modality'])
    outcome_col = find_col(d, ['outcome', 'task', 'endpoint'])
    frac_col = find_col(d, ['fraction', 'fraction_total', 'proportion'])
    d['modal_clean'] = d[modal_col].map(clean_modality_name)
    d['outcome_clean'] = d[outcome_col].map(clean_outcome_name)
    d['fraction_clean'] = pd.to_numeric(d[frac_col], errors='coerce')
    if d['fraction_clean'].max() > 1.5:
        d['fraction_clean'] = d['fraction_clean'] / 100.0
    order_modal = [m for m in MODALITY_ORDER_MRU if m in d['modal_clean'].unique()]
    extras = [m for m in d['modal_clean'].unique() if m not in order_modal]
    order_modal += extras
    fig, ax = plt.subplots(figsize=(7.2, 3.1), facecolor='white')
    y = np.arange(len(ENDPOINT_ORDER))
    left = np.zeros(len(ENDPOINT_ORDER), dtype=float)
    for modal in order_modal:
        vals = []
        for task in ENDPOINT_ORDER:
            ss = d[(d['outcome_clean'] == task) & (d['modal_clean'] == modal)]
            vals.append(float(ss['fraction_clean'].iloc[0]) if len(ss) else 0.0)
        vals = np.asarray(vals, dtype=float)
        c = MODALITY_COLORS.get(modal, '#777777')
        bars = ax.barh(y, vals, left=left, height=0.56, color=c, edgecolor='white', linewidth=0.8, label=modal, zorder=3)
        for i, (v, lft) in enumerate(zip(vals, left)):
            if v >= 0.11:
                ax.text(lft + v / 2, i, f'{100 * v:.0f}%', ha='center', va='center', fontsize=10.0, fontweight='bold', color='white', zorder=4)
        left += vals
    ax.set_yticks(y)
    ax.set_yticklabels(ENDPOINT_ORDER)
    ax.set_xlim(0, 1)
    ax.set_xticks(np.linspace(0, 1, 6))
    ax.set_xticklabels([f'{int(x * 100)}%' for x in np.linspace(0, 1, 6)])
    ax.set_xlabel('Fraction of total mean |SHAP|')
    ax.set_ylabel('')
    ax.set_ylim(len(ENDPOINT_ORDER) - 0.5, -0.5)
    clean_axes(ax, grid_axis='x')
    ax.legend(loc='center left', bbox_to_anchor=(1.015, 0.5), ncol=1, frameon=False, fontsize=10.0, borderaxespad=0.0)
    add_panel_header(fig, '6A', 'Distributed modality-level attribution across MRU endpoints')
    fig.subplots_adjust(left=0.16, right=0.78, top=0.8, bottom=0.14)
    d.to_csv(SOURCE_OUT_DIR / get_value('science.tables.attribution_031', 'attribution_031.csv'), index=False, encoding='utf-8-sig')
    save_figure(fig, MAIN_OUT_DIR / get_value('science.paths.directory_126', 'directory_126'))
    plt.show()
    return d


In [ ]:
def select_fig6b_features(n_shared=TOP_SHARED, n_preferred=TOP_PREFERRED_PER_ENDPOINT):
    selected = []
    group_map = {}
    shared = feature_preference.sort_values(['shared_score', 'overall_relative_importance'], ascending=False).head(n_shared)
    for f in shared['feature_clean']:
        if f not in selected:
            selected.append(f)
            group_map[f] = 'Shared backbone'
    for task in ENDPOINT_ORDER:
        col = f'{task}_preference_score'
        ss = feature_preference.sort_values([col, 'overall_relative_importance'], ascending=False)
        n_added = 0
        for f in ss['feature_clean']:
            if f in selected:
                continue
            if float(ss.loc[ss['feature_clean'] == f, col].iloc[0]) <= 0:
                continue
            selected.append(f)
            group_map[f] = f'{task}-preferred'
            n_added += 1
            if n_added >= n_preferred:
                break
    return (selected, group_map)
FIG6B_FEATURES, FIG6B_GROUP_MAP = select_fig6b_features()

def plot_fig6b_shared_preferred_map():
    selected = FIG6B_FEATURES.copy()
    d = determinant_long[determinant_long['feature_clean'].isin(selected)].copy()
    feature_modal = d[['feature_clean', 'modal_clean']].drop_duplicates('feature_clean').set_index('feature_clean')['modal_clean'].to_dict()
    group_order = ['Shared backbone', 'Response-preferred', 'PFS-preferred', 'OS-preferred']
    group_rank = {g: i for i, g in enumerate(group_order)}
    score_map = feature_preference.set_index('feature_clean')['overall_relative_importance'].to_dict()
    selected = sorted(selected, key=lambda f: (group_rank.get(FIG6B_GROUP_MAP.get(f, ''), 99), -score_map.get(f, 0)))
    grid = pd.MultiIndex.from_product([selected, ENDPOINT_ORDER], names=['feature_clean', 'outcome_clean']).to_frame(index=False).merge(d[['feature_clean', 'outcome_clean', 'modal_clean', 'relative_importance', 'direction_rho']], on=['feature_clean', 'outcome_clean'], how='left')
    grid['group'] = grid['feature_clean'].map(FIG6B_GROUP_MAP)
    grid['modal_clean'] = grid['modal_clean'].fillna(grid['feature_clean'].map(feature_modal))
    grid['relative_importance'] = grid['relative_importance'].fillna(0.0)
    max_imp = max(float(grid['relative_importance'].max()), 1e-06)

    def size_from_importance(v):
        if not np.isfinite(v) or v <= 0:
            return 15
        return 34 + 430 * (float(v) / max_imp) ** 0.8
    fig = plt.figure(figsize=(8.6, max(5.5, 0.34 * len(selected) + 1.8)), facecolor='white')
    gs = fig.add_gridspec(1, 3, width_ratios=[0.14, 0.58, 1.0], wspace=0.02)
    ax = fig.add_subplot(gs[0, 2])
    ax_label = fig.add_subplot(gs[0, 1], sharey=ax)
    ax_group = fig.add_subplot(gs[0, 0], sharey=ax)
    y_map = {f: i for i, f in enumerate(selected)}
    x_map = {t: j for j, t in enumerate(ENDPOINT_ORDER)}
    for _, r in grid.iterrows():
        x = x_map[r['outcome_clean']]
        y = y_map[r['feature_clean']]
        v = float(r['relative_importance'])
        rho = r['direction_rho']
        if v <= 0:
            ax.scatter(x, y, s=27, facecolor='white', edgecolor='#DADADA', linewidth=0.65, zorder=2)
        else:
            ax.scatter(x, y, s=size_from_importance(v), c=[rho if np.isfinite(rho) else 0.0], cmap=BENEFIT_CMAP, vmin=-1, vmax=1, edgecolor='white', linewidth=0.8, zorder=3)
    ax_label.set_xlim(0, 1.2)
    ax_label.set_ylim(len(selected) - 0.5, -0.5)
    ax_label.axis('off')
    for f in selected:
        y = y_map[f]
        modal = feature_modal.get(f, '')
        ax_label.scatter(0.04, y, s=48, marker='s', facecolor=MODALITY_COLORS.get(modal, '#AAAAAA'), edgecolor='white', linewidth=0.6, zorder=4)
        ax_label.text(0.11, y, clean_feature_label(f, 28), ha='left', va='center', rotation=45, rotation_mode='anchor', fontsize=8.8, color=COLOR_TEXT)
    ax_group.set_xlim(0, 1)
    ax_group.set_ylim(len(selected) - 0.5, -0.5)
    ax_group.axis('off')
    group_ranges = []
    for g in group_order:
        fs = [f for f in selected if FIG6B_GROUP_MAP.get(f) == g]
        if not fs:
            continue
        idxs = [y_map[f] for f in fs]
        group_ranges.append((g, min(idxs), max(idxs)))
    group_display = {'Shared backbone': 'Shared\nbackbone', 'Response-preferred': 'Response-\npreferred', 'PFS-preferred': 'PFS-\npreferred', 'OS-preferred': 'OS-\npreferred'}
    for g, lo, hi in group_ranges:
        if lo > 0:
            for aa in [ax_group, ax_label, ax]:
                aa.axhline(lo - 0.5, color='#D9D9D9', linewidth=0.75, zorder=1)
        ax_group.text(0.95, (lo + hi) / 2, group_display.get(g, g), ha='right', va='center', fontsize=9.7, fontweight='bold', color=COLOR_MUTED)
    ax.set_xticks(np.arange(len(ENDPOINT_ORDER)))
    ax.set_xticklabels(ENDPOINT_ORDER, fontsize=10.0, fontweight='bold')
    ax.set_yticks(np.arange(len(selected)))
    ax.set_yticklabels([])
    ax.set_xlim(-0.48, len(ENDPOINT_ORDER) - 0.52)
    ax.set_ylim(len(selected) - 0.5, -0.5)
    ax.tick_params(length=0)
    ax.grid(color=COLOR_GRID, linewidth=0.55, zorder=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=-1, vmax=1), cmap=BENEFIT_CMAP)
    sm.set_array([])
    cax = fig.add_axes([0.91, 0.22, 0.014, 0.52])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label('Direction of higher feature values\n(Spearman ρ with benefit-oriented SHAP)', fontsize=9.7)
    cbar.ax.tick_params(labelsize=6.3)
    cbar.outline.set_visible(False)
    modality_handles = [Line2D([0], [0], marker='s', linestyle='', markerfacecolor=MODALITY_COLORS[m], markeredgecolor='white', markersize=7.0, label=m) for m in MODALITY_ORDER_MRU if m in set(feature_modal.values())]
    fig.legend(handles=modality_handles, title='Feature modality', loc='upper center', bbox_to_anchor=(0.7, 0.925), ncol=max(1, len(modality_handles)), frameon=False, fontsize=9.7, title_fontsize=9.7)
    add_panel_header(fig, '6B', 'Shared and endpoint-preferred determinants within the final MRU projection')
    fig.subplots_adjust(left=0.045, right=0.885, top=0.86, bottom=0.07, wspace=0.02)
    grid.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_205', 'result_205.csv'), index=False, encoding='utf-8-sig')
    save_figure(fig, MAIN_OUT_DIR / get_value('science.paths.directory_127', 'directory_127'))
    plt.show()
    return grid


In [ ]:
def _barycentric_to_xy(p_response, p_pfs, p_os):
    vr = np.array([0.0, 0.0])
    vp = np.array([1.0, 0.0])
    vo = np.array([0.5, np.sqrt(3) / 2])
    p = np.array([p_response, p_pfs, p_os], dtype=float)
    xy = p[0] * vr + p[1] * vp + p[2] * vo
    return (float(xy[0]), float(xy[1]))

def plot_fig6c_endpoint_preference_simplex():
    d = feature_preference.copy()
    center = np.array([1 / 3, 1 / 3, 1 / 3], dtype=float)
    stretch = 1.85

    def _stretch_simplex(p):
        p = np.array(p, dtype=float)
        p2 = center + stretch * (p - center)
        p2 = np.clip(p2, 0, None)
        if p2.sum() <= 0:
            p2 = center.copy()
        else:
            p2 = p2 / p2.sum()
        return p2
    bary = d[['p_Response', 'p_PFS', 'p_OS']].astype(float).values
    bary2 = np.vstack([_stretch_simplex(p) for p in bary])
    d['pR_plot'] = bary2[:, 0]
    d['pP_plot'] = bary2[:, 1]
    d['pO_plot'] = bary2[:, 2]
    xy = d.apply(lambda r: _barycentric_to_xy(r['pR_plot'], r['pP_plot'], r['pO_plot']), axis=1)
    d['x'] = [v[0] for v in xy]
    d['y'] = [v[1] for v in xy]
    imp = d['overall_relative_importance'].astype(float)
    imp_min = float(imp.min())
    imp_max = float(imp.max())
    if imp_max > imp_min:
        size = 46 + 300 * ((imp - imp_min) / (imp_max - imp_min)) ** 0.75
    else:
        size = np.repeat(95.0, len(d))
    fig, ax = plt.subplots(figsize=(6.6, 5.8), facecolor='white')
    tri = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3) / 2], [0.0, 0.0]])
    ax.plot(tri[:, 0], tri[:, 1], color='#7A7A7A', linewidth=1.1, zorder=1)
    for frac in [0.25, 0.5, 0.75]:
        pts = [_barycentric_to_xy(frac, 1 - frac, 0), _barycentric_to_xy(frac, 0, 1 - frac)]
        ax.plot([pts[0][0], pts[1][0]], [pts[0][1], pts[1][1]], color=COLOR_GRID, linewidth=0.8, zorder=0)
        pts = [_barycentric_to_xy(1 - frac, frac, 0), _barycentric_to_xy(0, frac, 1 - frac)]
        ax.plot([pts[0][0], pts[1][0]], [pts[0][1], pts[1][1]], color=COLOR_GRID, linewidth=0.8, zorder=0)
        pts = [_barycentric_to_xy(1 - frac, 0, frac), _barycentric_to_xy(0, 1 - frac, frac)]
        ax.plot([pts[0][0], pts[1][0]], [pts[0][1], pts[1][1]], color=COLOR_GRID, linewidth=0.8, zorder=0)
    for modal in MODALITY_ORDER_MRU:
        ss = d.loc[d['modal_clean'] == modal].copy()
        if ss.empty:
            continue
        ax.scatter(ss['x'], ss['y'], s=size.loc[ss.index], facecolor=MODALITY_COLORS.get(modal, '#777777'), edgecolor='white', linewidth=0.9, alpha=0.96, label=modal, zorder=3)
    label_offsets = {'Response': (-14, -4), 'PFS': (14, -4), 'OS': (0, 14)}
    label_align = {'Response': ('right', 'top'), 'PFS': ('left', 'top'), 'OS': ('center', 'bottom')}
    used_features = set()
    label_rows = []
    for task in ENDPOINT_ORDER:
        score_col = f'{task}_preference_score'
        ss = d.sort_values([score_col, 'overall_relative_importance'], ascending=False).copy()
        ss = ss[pd.to_numeric(ss[score_col], errors='coerce').fillna(0) > 0]
        if ss.empty:
            continue
        for _, rr in ss.iterrows():
            feat = rr['feature_clean']
            if feat not in used_features:
                used_features.add(feat)
                label_rows.append((task, rr))
                break
    for task, rr in label_rows:
        dx, dy = label_offsets[task]
        ha, va = label_align[task]
        ax.annotate(clean_feature_label(rr['feature_clean'], 26), xy=(rr['x'], rr['y']), xytext=(dx, dy), textcoords='offset points', ha=ha, va=va, fontsize=8.8, color=COLOR_TEXT, bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.9), arrowprops=dict(arrowstyle='-', color='#A7A7A7', lw=0.65, shrinkA=1.5, shrinkB=2.0), zorder=5)
    cx, cy = _barycentric_to_xy(1 / 3, 1 / 3, 1 / 3)
    ax.text(-0.04, -0.04, 'Response', ha='right', va='top', fontsize=10.6, fontweight='bold', color=ENDPOINT_COLORS['Response'])
    ax.text(1.04, -0.04, 'PFS', ha='left', va='top', fontsize=10.6, fontweight='bold', color=ENDPOINT_COLORS['PFS'])
    ax.text(0.5, np.sqrt(3) / 2 + 0.05, 'OS', ha='center', va='bottom', fontsize=10.6, fontweight='bold', color=ENDPOINT_COLORS['OS'])
    ax.text(cx, cy - 0.05, 'Equal preference', ha='center', va='top', fontsize=9.0, color=COLOR_MUTED, bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.85))
    ax.set_xlim(-0.08, 1.08)
    ax.set_ylim(-0.08, np.sqrt(3) / 2 + 0.08)
    ax.set_aspect('equal')
    ax.axis('off')
    handles = [mpl.lines.Line2D([0], [0], marker='o', color='none', markerfacecolor=MODALITY_COLORS.get(m, '#777777'), markeredgecolor='none', markersize=9.2, label=m) for m in MODALITY_ORDER_MRU]
    ax.legend(handles=handles, frameon=False, loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=3, fontsize=8.8, columnspacing=1.2, handletextpad=0.5)
    fig.subplots_adjust(left=0.06, right=0.98, top=0.96, bottom=0.17)
    save_figure(fig, MAIN_OUT_DIR / get_value('science.paths.directory_128', 'directory_128'))
    plt.show()
    return d


In [ ]:
def load_mbp_annotations():
    if not MBP_PATH.exists():
        print('MBP table not found; patient heatmap will use prediction-only ordering.')
        return None
    d = pd.read_csv(MBP_PATH, encoding='utf-8-sig')
    sample_col = find_col(d, ['sample', 'sample_id', 'Sample ID', 'ID'], required=False)
    if sample_col is None:
        sample_col = d.columns[0]
    mbp_col = find_col(d, ['MBP_raw', 'MBP', 'MBP_group'], required=False)
    if mbp_col is None:
        print('MBP column not found; prediction-only ordering will be used.')
        return None
    out = pd.DataFrame({'sample': d[sample_col].map(normalize_sample_id), 'MBP_raw': d[mbp_col].astype(str)})
    out['MBP_raw'] = out['MBP_raw'].str.extract('(MBP-I{1,3})', expand=False)
    out = out.dropna(subset=['sample', 'MBP_raw']).drop_duplicates('sample').set_index('sample')
    return out

def build_patient_order():
    pred = pred_mru.copy()
    resp_col = find_col(pred, ['Response_pred', 'response_pred', 'Response_probability', 'response_probability'], required=False)
    pfs_col = find_col(pred, ['PFS_risk', 'pfs_risk'], required=False)
    os_col = find_col(pred, ['OS_risk', 'os_risk'], required=False)
    benefit_components = []
    if resp_col is not None:
        benefit_components.append(pred[resp_col].rank(pct=True).rename('Response_benefit_rank'))
    if pfs_col is not None:
        benefit_components.append((1.0 - pred[pfs_col].rank(pct=True)).rename('PFS_benefit_rank'))
    if os_col is not None:
        benefit_components.append((1.0 - pred[os_col].rank(pct=True)).rename('OS_benefit_rank'))
    if benefit_components:
        benefit_df = pd.concat(benefit_components, axis=1)
        pred['composite_benefit_rank'] = benefit_df.mean(axis=1)
    else:
        pred['composite_benefit_rank'] = np.linspace(0, 1, len(pred))
    mbp = load_mbp_annotations()
    if mbp is not None:
        pred = pred.join(mbp, how='left')
        mbp_rank = {'MBP-I': 0, 'MBP-II': 1, 'MBP-III': 2}
        pred['MBP_order'] = pred['MBP_raw'].map(mbp_rank).fillna(99)
        pred = pred.sort_values(['MBP_order', 'composite_benefit_rank'], ascending=[True, False])
    else:
        pred['MBP_raw'] = np.nan
        pred = pred.sort_values('composite_benefit_rank', ascending=False)
    return pred
patient_order_df = build_patient_order()
PATIENT_ORDER = patient_order_df.index.tolist()
display(patient_order_df[[c for c in ['MBP_raw', 'composite_benefit_rank'] if c in patient_order_df.columns]].head())


In [ ]:
def _row_scale_with_reference(M, quantile=0.95):
    M = M.astype(float).copy()
    denom = {}
    out = M.copy()
    for idx in M.index:
        vals = M.loc[idx].values.astype(float)
        finite = np.abs(vals[np.isfinite(vals)])
        if len(finite) == 0:
            d = 1.0
        else:
            d = float(np.quantile(finite, quantile))
            if not np.isfinite(d) or d <= 1e-12:
                d = float(np.nanmax(finite)) if len(finite) else 1.0
            if not np.isfinite(d) or d <= 1e-12:
                d = 1.0
        denom[idx] = d
        out.loc[idx] = np.clip(vals / d, -1, 1)
    return (out, pd.Series(denom))

def plot_fig6d_patient_landscape(max_features=15):
    feature_score = feature_preference.set_index('feature_clean')['overall_relative_importance'].sort_values(ascending=False)
    features = feature_score.head(max_features).index.tolist()
    feature_modal = feature_preference[['feature_clean', 'modal_clean']].drop_duplicates('feature_clean').set_index('feature_clean')['modal_clean'].to_dict()
    modality_rank = {m: i for i, m in enumerate(MODALITY_ORDER_MRU)}
    features = sorted(features, key=lambda f: (modality_rank.get(feature_modal.get(f, ''), 99), -feature_score.get(f, 0)))
    common_patients = set(PATIENT_ORDER)
    for task in ENDPOINT_ORDER:
        common_patients &= set(shap_benefit[task].index)
    patients = [p for p in PATIENT_ORDER if p in common_patients]
    if not patients:
        raise ValueError('No common patient IDs across predictions and SHAP matrices.')
    mbp_series = patient_order_df.reindex(patients)['MBP_raw'] if 'MBP_raw' in patient_order_df.columns else pd.Series(index=patients, dtype=object)
    n_patients = len(patients)
    fig = plt.figure(figsize=(13.8, max(5.8, 0.34 * len(features) + 2.4)), facecolor='white')
    gs = fig.add_gridspec(nrows=2, ncols=6, height_ratios=[0.12, 1.0], width_ratios=[1.0, 0.22, 1.0, 0.22, 1.0, 0.22], hspace=0.06, wspace=0.08)
    patient_axes = []
    group_axes = []
    im = None
    rgba = np.ones((1, n_patients, 4), dtype=float)
    for j, p in enumerate(patients):
        grp = mbp_series.get(p, np.nan)
        rgba[0, j, :] = mpl.colors.to_rgba(MBP_COLORS.get(grp, '#E6E6E6'))
    for k, task in enumerate(ENDPOINT_ORDER):
        col_heat = 2 * k
        col_group = col_heat + 1
        ax_top = fig.add_subplot(gs[0, col_heat])
        ax = fig.add_subplot(gs[1, col_heat])
        ax_g = fig.add_subplot(gs[1, col_group])
        patient_axes.append(ax)
        group_axes.append(ax_g)
        ax_top.imshow(rgba, aspect='auto', interpolation='nearest')
        ax_top.set_xticks([])
        ax_top.set_yticks([])
        for spine in ax_top.spines.values():
            spine.set_visible(False)
        available = [f for f in features if f in shap_benefit[task].columns]
        M = shap_benefit[task].loc[patients, available].T
        M = M.reindex(features).fillna(0.0)
        M_scaled, row_denom = _row_scale_with_reference(M, quantile=0.95)
        im = ax.imshow(M_scaled.values, aspect='auto', interpolation='nearest', cmap=BENEFIT_CMAP, vmin=-1, vmax=1)
        ax.set_title(task, fontsize=10.9, fontweight='bold', color=ENDPOINT_COLORS[task], pad=6)
        ax.set_xticks([])
        ax.set_yticks(np.arange(len(features)))
        if k == 0:
            ax.set_yticklabels([clean_feature_label(f, 30) for f in features], fontsize=8.7)
            for lab in ax.get_yticklabels():
                lab.set_rotation(45)
                lab.set_rotation_mode('anchor')
                lab.set_ha('right')
                lab.set_va('center')
        else:
            ax.set_yticklabels([])
        ax.tick_params(length=0, axis='y', pad=2)
        for spine in ax.spines.values():
            spine.set_visible(False)
        if mbp_series.notna().any():
            groups = mbp_series.values
            for j in range(1, len(groups)):
                if groups[j] != groups[j - 1]:
                    ax.axvline(j - 0.5, color='white', linewidth=1.15, zorder=4)
                    ax_top.axvline(j - 0.5, color='white', linewidth=1.15, zorder=4)
        group_mean = pd.DataFrame(index=features, columns=MBP_ORDER, dtype=float)
        if mbp_series.notna().any():
            raw = shap_benefit[task].reindex(index=patients, columns=features)
            for grp in MBP_ORDER:
                ids = mbp_series[mbp_series == grp].index
                ids = [x for x in ids if x in raw.index]
                if ids:
                    group_mean[grp] = raw.loc[ids].mean(axis=0).reindex(features)
            for f in features:
                d = float(row_denom.get(f, 1.0))
                group_mean.loc[f] = np.clip(pd.to_numeric(group_mean.loc[f], errors='coerce') / d, -1, 1)
            ax_g.imshow(group_mean.astype(float).values, aspect='auto', interpolation='nearest', cmap=BENEFIT_CMAP, vmin=-1, vmax=1)
            ax_g.set_xticks(np.arange(len(MBP_ORDER)))
            ax_g.set_xticklabels(['I', 'II', 'III'], fontsize=9.2, rotation=90)
        else:
            ax_g.text(0.5, 0.5, 'MBP\nN/A', ha='center', va='center', fontsize=9.2, color=COLOR_MUTED, transform=ax_g.transAxes)
            ax_g.set_xticks([])
        ax_g.set_yticks([])
        ax_g.tick_params(length=0)
        for spine in ax_g.spines.values():
            spine.set_visible(False)
        if k == 2:
            ax_g.set_title('Mean\nby MBP', fontsize=9.2, color=COLOR_MUTED, pad=5)
        M_scaled.to_csv(SOURCE_OUT_DIR / f"{get_value('science.paths.directory_129', 'directory_129')}{task}.csv", encoding='utf-8-sig')
        group_mean.to_csv(SOURCE_OUT_DIR / f"{get_value('science.paths.directory_130', 'directory_130')}{task}.csv", encoding='utf-8-sig')
    ax0 = patient_axes[0]
    for i, f in enumerate(features):
        modal = feature_modal.get(f, '')
        ax0.scatter(-3.0, i, s=45, marker='s', facecolor=MODALITY_COLORS.get(modal, '#AAAAAA'), edgecolor='white', linewidth=0.5, clip_on=False, zorder=5)
    cbar = fig.colorbar(im, ax=patient_axes + group_axes, orientation='vertical', fraction=0.014, pad=0.012)
    cbar.set_label('Row-scaled local SHAP\n(benefit-oriented)', fontsize=9.7)
    cbar.ax.tick_params(labelsize=7.0)
    cbar.outline.set_visible(False)
    if mbp_series.notna().any():
        mbp_handles = [Line2D([0], [0], marker='s', linestyle='', markerfacecolor=MBP_COLORS[g], markeredgecolor='white', markersize=7.5, label=g) for g in MBP_ORDER if g in set(mbp_series.dropna())]
        fig.legend(handles=mbp_handles, loc='upper center', bbox_to_anchor=(0.53, 0.965), ncol=len(mbp_handles), frameon=False, fontsize=9.2, title='Patient order / MBP', title_fontsize=9.5)
    add_panel_header(fig, '6D', 'Patient-level heterogeneity in MRU attribution')
    pd.DataFrame({'sample': patients, 'MBP_raw': patient_order_df.reindex(patients)['MBP_raw'].values if 'MBP_raw' in patient_order_df.columns else np.nan, 'composite_benefit_rank': patient_order_df.reindex(patients)['composite_benefit_rank'].values}).to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_206', 'result_206.csv'), index=False, encoding='utf-8-sig')
    fig.subplots_adjust(left=0.33, right=0.94, top=0.83, bottom=0.08)
    save_figure(fig, MAIN_OUT_DIR / get_value('science.paths.directory_131', 'directory_131'))
    plt.show()
    return (features, patients)


In [ ]:
def _percentile_scale(v):
    s = pd.Series(v, dtype=float)
    ranks = s.rank(method='average', pct=True)
    return ranks.values

def _collect_beeswarm_data(task, features, seed=100):
    rng = np.random.default_rng(seed)
    rows = []
    for yi, feat in enumerate(features):
        if feat not in shap_benefit[task].columns or feat not in X_mru.columns:
            continue
        common = shap_benefit[task].index.intersection(X_mru.index)
        shap_vals = pd.to_numeric(shap_benefit[task].loc[common, feat], errors='coerce').values.astype(float)
        feat_vals = pd.to_numeric(X_mru.loc[common, feat], errors='coerce').values.astype(float)
        mask = np.isfinite(shap_vals) & np.isfinite(feat_vals)
        shap_vals = shap_vals[mask]
        feat_vals = feat_vals[mask]
        sample_ids = np.asarray(common)[mask]
        if len(shap_vals) == 0:
            continue
        value_pct = _percentile_scale(feat_vals)
        jitter = rng.normal(0, 0.075, size=len(shap_vals))
        jitter = np.clip(jitter, -0.18, 0.18)
        for sid, sv, fv, vp, jj in zip(sample_ids, shap_vals, feat_vals, value_pct, jitter):
            rows.append({'sample': sid, 'outcome': task, 'feature': feat, 'feature_value': float(fv), 'feature_value_percentile': float(vp), 'benefit_oriented_shap': float(sv), 'y_base': yi, 'y_jitter': float(jj)})
    return pd.DataFrame(rows)

def plot_fig6e_global_beeswarm(top_n=10):
    fig, axes = plt.subplots(3, 1, figsize=(6.0, max(9.0, 1.2 * top_n + 3.0)), facecolor='white', sharex=False, sharey=False)
    axes = np.atleast_1d(axes)
    source_frames = []
    for ai, (ax, task) in enumerate(zip(axes, ENDPOINT_ORDER)):
        imp_task = determinant_long[determinant_long['outcome_clean'] == task].sort_values('mean_abs_shap_clean', ascending=False).head(top_n).copy()
        features = imp_task['feature_clean'].tolist()[::-1]
        dd = _collect_beeswarm_data(task=task, features=features, seed=100 + ENDPOINT_ORDER.index(task))
        source_frames.append(dd)
        if not dd.empty:
            ax.scatter(dd['benefit_oriented_shap'], dd['y_base'] + dd['y_jitter'], c=dd['feature_value_percentile'], cmap=FEATURE_VALUE_CMAP, vmin=0, vmax=1, s=18, alpha=0.92, edgecolors='none', rasterized=True, zorder=3)
        ax.axvline(0, color='#808080', linewidth=0.85, linestyle=(0, (3, 3)), zorder=1)
        ax.set_yticks(np.arange(len(features)))
        ax.set_yticklabels([clean_feature_label(f, 26) for f in features], fontsize=8.7)
        for lab in ax.get_yticklabels():
            lab.set_rotation(45)
            lab.set_rotation_mode('anchor')
            lab.set_ha('right')
            lab.set_va('center')
        ax.set_title(task, fontsize=11.0, fontweight='bold', color=ENDPOINT_COLORS[task], loc='right', pad=4)
        ax.set_xlabel('Benefit-oriented SHAP value' if ai == len(ENDPOINT_ORDER) - 1 else '')
        ax.set_ylabel('')
        ax.tick_params(axis='y', length=0, pad=2)
        clean_axes(ax, grid_axis='x')
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0, vmax=1), cmap=FEATURE_VALUE_CMAP)
    sm.set_array([])
    cax = fig.add_axes([0.905, 0.18, 0.018, 0.64])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(['Low', 'High'])
    cbar.set_label('Feature value', fontsize=9.2)
    cbar.ax.tick_params(labelsize=8.2)
    cbar.outline.set_visible(False)
    source_df = pd.concat(source_frames, ignore_index=True)
    source_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_207', 'result_207.csv'), index=False, encoding='utf-8-sig')
    fig.subplots_adjust(left=0.31, right=0.87, top=0.97, bottom=0.07, hspace=0.34)
    save_figure(fig, MAIN_OUT_DIR / get_value('science.paths.directory_132', 'directory_132'))
    plt.show()
    return source_df

def plot_fig6e_global_beeswarm_triptych_sharedy(top_n=10):
    common_order = feature_preference.sort_values('overall_relative_importance', ascending=False)['feature_clean'].tolist()
    common_order = [f for f in common_order if f in X_mru.columns and all((f in shap_benefit[t].columns for t in ENDPOINT_ORDER))][:top_n]
    plot_features = common_order[::-1]
    fig, axes = plt.subplots(1, 3, figsize=(10.6, max(4.8, 0.42 * len(plot_features) + 1.8)), facecolor='white', sharey=True)
    axes = np.atleast_1d(axes)
    source_frames = []
    for ai, (ax, task) in enumerate(zip(axes, ENDPOINT_ORDER)):
        dd = _collect_beeswarm_data(task=task, features=plot_features, seed=300 + ai)
        source_frames.append(dd)
        if not dd.empty:
            ax.scatter(dd['benefit_oriented_shap'], dd['y_base'] + dd['y_jitter'], c=dd['feature_value_percentile'], cmap=FEATURE_VALUE_CMAP, vmin=0, vmax=1, s=16, alpha=0.9, edgecolors='none', rasterized=True, zorder=3)
        ax.axvline(0, color='#808080', linewidth=0.85, linestyle=(0, (3, 3)), zorder=1)
        ax.set_title(task, fontsize=10.8, fontweight='bold', color=ENDPOINT_COLORS[task], pad=4)
        ax.set_xlabel('Benefit-oriented SHAP value')
        ax.set_yticks(np.arange(len(plot_features)))
        if ai == 0:
            ax.set_yticklabels([clean_feature_label(f, 26) for f in plot_features], fontsize=8.6)
            for lab in ax.get_yticklabels():
                lab.set_rotation(45)
                lab.set_rotation_mode('anchor')
                lab.set_ha('right')
                lab.set_va('center')
        else:
            ax.tick_params(axis='y', left=False, labelleft=False)
        clean_axes(ax, grid_axis='x')
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0, vmax=1), cmap=FEATURE_VALUE_CMAP)
    sm.set_array([])
    cax = fig.add_axes([0.915, 0.2, 0.014, 0.6])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(['Low', 'High'])
    cbar.set_label('Feature value', fontsize=9.0)
    cbar.ax.tick_params(labelsize=8.0)
    cbar.outline.set_visible(False)
    source_df = pd.concat(source_frames, ignore_index=True)
    source_df.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_208', 'result_208.csv'), index=False, encoding='utf-8-sig')
    fig.subplots_adjust(left=0.22, right=0.89, top=0.91, bottom=0.14, wspace=0.18)
    save_figure(fig, MAIN_OUT_DIR / get_value('science.paths.directory_133', 'directory_133'))
    plt.show()
    return source_df


In [ ]:
def _spearman_pair(a, b):
    a = pd.Series(a, dtype=float)
    b = pd.Series(b, dtype=float)
    mask = a.notna() & b.notna()
    if mask.sum() < 3:
        return np.nan
    return a[mask].rank().corr(b[mask].rank(), method='pearson')

def build_patient_cross_endpoint_concordance():
    common_features = set(shap_benefit['Response'].columns)
    for task in ['PFS', 'OS']:
        common_features &= set(shap_benefit[task].columns)
    common_features = [f for f in X_mru.columns if f in common_features]
    common_patients = set(shap_benefit['Response'].index)
    for task in ['PFS', 'OS']:
        common_patients &= set(shap_benefit[task].index)
    ordered_patients = [p for p in PATIENT_ORDER if p in common_patients]
    if not ordered_patients:
        ordered_patients = sorted(common_patients)
    pairs = [('Response', 'PFS'), ('Response', 'OS'), ('PFS', 'OS')]
    rows = []
    for patient in ordered_patients:
        for a, b in pairs:
            rho = _spearman_pair(shap_benefit[a].loc[patient, common_features].values, shap_benefit[b].loc[patient, common_features].values)
            grp = patient_order_df.loc[patient, 'MBP_raw'] if patient in patient_order_df.index and 'MBP_raw' in patient_order_df.columns else np.nan
            rows.append({'sample': patient, 'pair': f'{a}–{b}', 'endpoint_a': a, 'endpoint_b': b, 'spearman_rho': rho, 'MBP_raw': grp, 'n_features': len(common_features)})
    d = pd.DataFrame(rows)
    d.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_209', 'result_209.csv'), index=False, encoding='utf-8-sig')
    return d
patient_concordance = build_patient_cross_endpoint_concordance()

def plot_fig6f_patient_concordance():
    d = patient_concordance.copy()
    pair_order = ['Response–PFS', 'Response–OS', 'PFS–OS']
    fig, ax = plt.subplots(figsize=(5.9, 3.7), facecolor='white')
    rng = np.random.default_rng(20260810)
    for i, pair in enumerate(pair_order):
        vals = d.loc[d['pair'] == pair, 'spearman_rho'].dropna().astype(float).values
        if len(vals) == 0:
            continue
        vp = ax.violinplot(vals, positions=[i], widths=0.52, showmeans=False, showmedians=False, showextrema=False)
        for body in vp['bodies']:
            body.set_facecolor('#D8D8D8')
            body.set_edgecolor('#B0B0B0')
            body.set_linewidth(0.8)
            body.set_alpha(0.85)
        q1, med, q3 = np.percentile(vals, [25, 50, 75])
        ax.plot([i - 0.14, i + 0.14], [med, med], color='#6F6F6F', linewidth=1.5, zorder=4)
        ax.plot([i, i], [q1, q3], color='#6F6F6F', linewidth=1.3, zorder=4)
        ss = d.loc[d['pair'] == pair].copy()
        if 'MBP_raw' in ss.columns:
            for g in ['MBP-I', 'MBP-II', 'MBP-III']:
                sg = ss.loc[ss['MBP_raw'] == g].copy()
                if sg.empty:
                    continue
                jitter = rng.uniform(-0.12, 0.12, size=len(sg))
                ax.scatter(np.full(len(sg), i) + jitter, sg['spearman_rho'].astype(float), s=14, color=MBP_COLORS.get(g, '#888888'), edgecolor='white', linewidth=0.3, alpha=0.9, zorder=5, label=g if i == 0 else None)
        else:
            jitter = rng.uniform(-0.12, 0.12, size=len(ss))
            ax.scatter(np.full(len(ss), i) + jitter, ss['spearman_rho'].astype(float), s=14, color='#7A7A7A', edgecolor='white', linewidth=0.3, alpha=0.88, zorder=5)
        ax.text(i, min(1.008, med + 0.035), f'median ρ={med:.2f}', ha='center', va='bottom', fontsize=8.3, color=COLOR_TEXT, bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.88), zorder=6)
    ax.set_xticks(np.arange(len(pair_order)))
    ax.set_xticklabels(pair_order)
    ax.set_ylabel('Within-patient Spearman correlation\nacross MRU feature SHAP profiles')
    ax.set_ylim(0.68, 1.02)
    ax.set_yticks([0.7, 0.8, 0.9, 1.0])
    clean_axes(ax, grid_axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if 'MBP_raw' in d.columns:
        handles = [mpl.lines.Line2D([0], [0], marker='o', linestyle='none', markersize=5.5, markerfacecolor=MBP_COLORS[g], markeredgecolor='none', label=g) for g in ['MBP-I', 'MBP-II', 'MBP-III']]
        ax.legend(handles=handles, frameon=False, title='Patient group', fontsize=8.2, title_fontsize=8.5, loc='lower left', bbox_to_anchor=(0.01, 0.02), ncol=1, handletextpad=0.4, borderaxespad=0.0)
    fig.subplots_adjust(left=0.16, right=0.98, top=0.94, bottom=0.18)
    save_figure(fig, MAIN_OUT_DIR / get_value('science.paths.directory_134', 'directory_134'))
    plt.show()
    return d


In [ ]:
def plot_fig4def_triptych_sharedy():
    available_modalities = set(stage1_perf['modal_clean'].dropna().astype(str).unique()) | set(golden['modal_clean'].dropna().astype(str).unique()) | set(stage1_stability_feature['modal_clean'].dropna().astype(str).unique())
    order = [m for m in MODALITY_ORDER_STAGE1 if m in available_modalities]
    order += sorted(available_modalities - set(order))
    if len(order) != len(set(order)):
        raise RuntimeError(f'Duplicate modality labels detected in Figure 4 triptych order: {order}')
    fig = plt.figure(figsize=(13.8, 3.45), facecolor='white')
    gs = fig.add_gridspec(1, 3, width_ratios=[1.08, 1.1, 1.4], wspace=0.18)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1], sharey=ax1)
    ax3 = fig.add_subplot(gs[0, 2], sharey=ax1)
    raw_counts = stage1_perf[['modal_clean', perf_nfeat_col]].drop_duplicates('modal_clean').rename(columns={perf_nfeat_col: 'input_features'})
    retained_counts = golden.groupby('modal_clean')['feature_clean'].nunique().rename('retained_features').reset_index()
    d1 = raw_counts.merge(retained_counts, on='modal_clean', how='inner')
    d1['modal_clean'] = d1['modal_clean'].astype(str)
    d1 = d1.set_index('modal_clean').reindex(order).reset_index()
    y_map = {m: i for i, m in enumerate(order)}
    for _, r in d1.dropna(subset=['input_features', 'retained_features']).iterrows():
        modal = str(r['modal_clean'])
        y = y_map[modal]
        raw = float(r['input_features'])
        kept = float(r['retained_features'])
        color = MODALITY_COLORS.get(modal, '#808080')
        ax1.plot([kept, raw], [y, y], color='#B7B7B7', linewidth=1.4, zorder=1)
        ax1.scatter(kept, y, s=72, facecolor=color, edgecolor='white', linewidth=0.8, zorder=3)
        ax1.scatter(raw, y, s=78, facecolor='white', edgecolor='#7A7A7A', linewidth=1.0, zorder=2)
        ax1.text(kept * 1.1, y, f'{int(kept):,}', ha='left', va='center', fontsize=8.8, color=color, fontweight='bold')
        ax1.text(raw * 1.03, y, f'{int(raw):,}', ha='left', va='center', fontsize=8.6, color='#666666')
    ax1.set_xscale('log')
    ax1.set_xlabel('Number of features (log scale)')
    ax1.set_ylabel('')
    ax1.set_yticks(np.arange(len(order)))
    ax1.set_yticklabels(order)
    ax1.set_ylim(len(order) - 0.5, -0.5)
    clean_axes(ax1, grid_axis='x')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    handles_1 = [Line2D([0], [0], marker='o', linestyle='', markerfacecolor='white', markeredgecolor='#7A7A7A', markersize=7.2, label='Input features'), Line2D([0], [0], marker='o', linestyle='', markerfacecolor='#222222', markeredgecolor='white', markersize=7.2, label='SHAP-retained')]
    ax1.legend(handles=handles_1, frameon=False, loc='lower left', bbox_to_anchor=(0.42, 0.02), ncol=2, columnspacing=1.5, handletextpad=0.6)
    counts = golden.groupby(['modal_clean', 'pattern'])['feature_clean'].nunique().rename('n_features').reset_index()
    counts = counts.groupby(['modal_clean', 'pattern'], as_index=False)['n_features'].sum()
    d2 = pd.MultiIndex.from_product([order, PATTERN_ORDER], names=['modal_clean', 'pattern']).to_frame(index=False).merge(counts, on=['modal_clean', 'pattern'], how='left', validate='one_to_one')
    d2['n_features'] = d2['n_features'].fillna(0).astype(int)
    pattern_labels = {'Response-specific': 'Response\nspecific', 'PFS-specific': 'PFS\nspecific', 'OS-specific': 'OS\nspecific', 'Shared by 2': 'Shared by\n2 endpoints', 'Shared by 3': 'Shared by\n3 endpoints'}
    pattern_rank = {p: j for j, p in enumerate(PATTERN_ORDER)}
    for i, modal in enumerate(order):
        ss = d2.loc[d2['modal_clean'] == modal].copy()
        ss['pattern_rank'] = ss['pattern'].map(pattern_rank)
        ss = ss.sort_values('pattern_rank').reset_index(drop=True)
        if ss['pattern'].duplicated().any():
            raise RuntimeError(f"Duplicate endpoint-pattern rows remain for modality {modal}: {ss.loc[ss['pattern'].duplicated(keep=False), 'pattern'].tolist()}")
        for j, (_, rr) in enumerate(ss.iterrows()):
            n = int(rr['n_features'])
            if n <= 0:
                continue
            color = MODALITY_COLORS.get(modal, '#808080')
            size = 95 + 78 * np.sqrt(n)
            ax2.scatter(j, i, s=size, facecolor=color, edgecolor='white', linewidth=0.9, alpha=0.96, zorder=3)
            ax2.text(j, i, str(n), ha='center', va='center', fontsize=9.1, color='white', fontweight='bold', zorder=4)
    ax2.set_xticks(np.arange(len(PATTERN_ORDER)))
    ax2.set_xticklabels([pattern_labels.get(p, p) for p in PATTERN_ORDER])
    ax2.set_xlim(-0.55, len(PATTERN_ORDER) - 0.45)
    ax2.set_ylim(len(order) - 0.5, -0.5)
    ax2.tick_params(axis='y', left=False, labelleft=False, length=0)
    clean_axes(ax2, grid_axis='both')
    for spine in ax2.spines.values():
        spine.set_visible(False)
    d3 = stage1_stability_feature.copy()
    if 'pattern' not in d3.columns and 'selected_outcomes' in d3.columns:
        d3['pattern'] = d3['selected_outcomes']
    for modal in order:
        ss = d3[d3['modal_clean'] == modal].copy()
        if ss.empty:
            continue
        ss = ss.sort_values('mean_topk_frequency').reset_index(drop=True)
        y0 = y_map[modal]
        y = y0 + np.linspace(-0.13, 0.13, len(ss)) if len(ss) > 1 else np.array([y0])
        sizes = 26 + 24 * ss['n_selected_endpoints'].clip(1, 3).values
        color = MODALITY_COLORS.get(modal, '#808080')
        ax3.scatter(ss['mean_topk_frequency'], y, s=sizes, facecolor=color, edgecolor='white', linewidth=0.65, alpha=0.96, zorder=3)
        med = float(ss['mean_topk_frequency'].median())
        q25 = float(ss['mean_topk_frequency'].quantile(0.25))
        q75 = float(ss['mean_topk_frequency'].quantile(0.75))
        ax3.plot([q25, q75], [y0, y0], color='#4F4F4F', linewidth=2.0, solid_capstyle='round', zorder=2)
        ax3.scatter(med, y0, s=58, facecolor='white', edgecolor='#4F4F4F', linewidth=1.0, zorder=4)
    ax3.axvline(0.5, color='#9A9A9A', linewidth=0.85, linestyle=(0, (3, 3)), zorder=1)
    ax3.set_xlim(-0.02, 1.02)
    ax3.set_xticks(np.linspace(0, 1, 6))
    ax3.set_xticklabels([f'{int(x * 100)}%' for x in np.linspace(0, 1, 6)])
    ax3.set_xlabel('Bootstrap Top-k retention frequency')
    ax3.set_ylim(len(order) - 0.5, -0.5)
    ax3.tick_params(axis='y', left=False, labelleft=False, length=0)
    clean_axes(ax3, grid_axis='x')
    handles_3 = [ax3.scatter([], [], s=26 + 24 * k, facecolor='#777777', edgecolor='white', linewidth=0.6, label=f'{k} endpoint' + ('s' if k > 1 else '')) for k in [1, 2, 3]]
    ax3.legend(handles=handles_3, title='Originally selected in', frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8.3, title_fontsize=8.6, handletextpad=0.7, borderaxespad=0.0)
    fig.subplots_adjust(left=0.12, right=0.88, top=0.92, bottom=0.23)
    save_figure(fig, MAIN_OUT_DIR / get_value('science.paths.directory_135', 'directory_135'))
    source_bundle = {'feature_compression': d1, 'endpoint_overlap': d2, 'bootstrap_stability': d3}
    d1.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_210', 'result_210.csv'), index=False, encoding='utf-8-sig')
    d2.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_211', 'result_211.csv'), index=False, encoding='utf-8-sig')
    d3.to_csv(SOURCE_OUT_DIR / get_value('science.tables.result_212', 'result_212.csv'), index=False, encoding='utf-8-sig')
    plt.show()
    return source_bundle


In [ ]:
outputs = {}
outputs[get_value('science.paths.directory_136', 'directory_136')] = plot_fig4_feature_compression()
outputs[get_value('science.paths.directory_137', 'directory_137')] = plot_fig4_endpoint_overlap()
outputs[get_value('science.paths.directory_138', 'directory_138')] = plot_fig4f_bootstrap_stability()
outputs[get_value('science.paths.directory_139', 'directory_139')] = plot_fig4def_triptych_sharedy()
outputs[get_value('science.paths.directory_140', 'directory_140')] = plot_fig6a_modality_attribution()
outputs[get_value('science.paths.directory_141', 'directory_141')] = plot_fig6b_shared_preferred_map()
outputs[get_value('science.paths.directory_142', 'directory_142')] = plot_fig6c_endpoint_preference_simplex()
outputs[get_value('science.paths.directory_143', 'directory_143')] = plot_fig6d_patient_landscape(max_features=MAX_PATIENT_HEATMAP_FEATURES)
outputs[get_value('science.paths.directory_144', 'directory_144')] = plot_fig6e_global_beeswarm(top_n=10)
outputs[get_value('science.paths.directory_145', 'directory_145')] = plot_fig6e_global_beeswarm_triptych_sharedy(top_n=10)
outputs[get_value('science.paths.directory_146', 'directory_146')] = plot_fig6f_patient_concordance()





In [ ]:
from sklearn.preprocessing import MinMaxScaler
try:
    from pycirclize import Circos
except ImportError as e:
    raise ImportError('pycirclize is required for the supplementary Circos panels. Install it with: pip install pycirclize') from e
SUPP_OUT_DIR = OUT_DIR / 'Supplement'
CIRCOS_OUT_DIR = SUPP_OUT_DIR / get_value('science.paths.directory_147', 'directory_147')
CIRCOS_SOURCE_DIR = SOURCE_OUT_DIR / get_value('science.paths.directory_147', 'directory_147')
for d in [SUPP_OUT_DIR, CIRCOS_OUT_DIR, CIRCOS_SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
CIRCOS_MODALITIES = ['Metagenomics', 'Radiomics', 'Blood', 'Urine']
CIRCOS_TASKS = ['Response', 'PFS', 'OS']
CIRCOS_CANVAS_SIZE = (9.2, 9.2)
CIRCOS_SIZE = [6, 18, 6, 14]
TRACK4_STEP = 5.5
FONT_CENTER = 24
FONT_SECTOR = 12
FONT_LABEL_L1 = 7.0
FONT_LABEL_L2 = 5.8
SHARED_COLORS = ['#00A087', '#E6A01A']
OBSERVED_COLOR = '#6F6F6F'
BARCODE_POS_COLOR = '#5B8BD1'
BARCODE_NEG_COLOR = '#F9FAFC'
PANEL_LABELS = {'Metagenomics': 'SxB1', 'Radiomics': 'SxB2', 'Blood': 'SxB3', 'Urine': 'SxB4'}

def _circos_normalize_sample_id(x):
    return str(x).lower().strip()

def _circos_read_indexed(path, compression=None):
    df = pd.read_csv(path, index_col=0, encoding='utf-8-sig', compression=compression)
    df.index = df.index.map(_circos_normalize_sample_id)
    return df

def _circos_minmax_01(arr, jitter_binary=False, seed=42):
    arr = np.asarray(arr, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    arr2 = arr.copy()
    if jitter_binary:
        uniq = np.unique(arr2[finite])
        if len(uniq) <= 2:
            rng = np.random.default_rng(seed)
            arr2[finite] = arr2[finite] + rng.uniform(-0.08, 0.08, size=finite.sum())
    out = np.full(len(arr2), np.nan, dtype=float)
    scaler = MinMaxScaler()
    out[finite] = scaler.fit_transform(arr2[finite].reshape(-1, 1)).ravel()
    return out

def _circos_clean_feature_name(feat, max_len=18):
    s = str(feat)
    if '__' in s:
        s = s.split('__')[-1]
    for prefix in ['g__', 's__', 'k__']:
        s = s.replace(prefix, '')
    if len(s) > max_len:
        s = s[:max_len - 1] + '…'
    return s

def _circos_display_modal_name(modal_name):
    return modal_name

def _circos_input_dir(modal_name):
    return STAGE1_DIR / modal_name / get_value('science.paths.directory_004', 'directory_004')

def _circos_signed_path(modal_name, task):
    return _circos_input_dir(modal_name) / f'{modal_name}_cv_shap_signed_{task}.csv.gz'

def _circos_abs_path(modal_name, task):
    return _circos_input_dir(modal_name) / f'{modal_name}_cv_shap_abs_{task}.csv.gz'

def _sector_heatmap_cmap(task):
    task_color = ENDPOINT_COLORS[task]
    return mpl.colors.LinearSegmentedColormap.from_list(f'{task}_light', ['#FFFFFF', '#F6F8FB', task_color])

def _barcode_cmap():
    return mpl.colors.LinearSegmentedColormap.from_list('barcode_lightblue', [BARCODE_NEG_COLOR, BARCODE_POS_COLOR])

def build_current_stage1_circos_feature_table():
    records = []
    for modal_name in CIRCOS_MODALITIES:
        shap_abs = {}
        for task in CIRCOS_TASKS:
            p = _circos_abs_path(modal_name, task)
            if not p.exists():
                raise FileNotFoundError(f'Missing Stage-1 OOF SHAP file:\n{p}')
            shap_abs[task] = _circos_read_indexed(p, compression='gzip')
        common_features = set(shap_abs['Response'].columns)
        common_features &= set(shap_abs['PFS'].columns)
        common_features &= set(shap_abs['OS'].columns)
        common_features = list(common_features)
        if len(common_features) == 0:
            raise ValueError(f'No common Stage-1 SHAP features for {modal_name}.')
        mean_abs_by_task = {task: shap_abs[task][common_features].mean(axis=0) for task in CIRCOS_TASKS}
        mean_global = pd.concat(mean_abs_by_task, axis=1)
        mean_global.columns = CIRCOS_TASKS
        mean_global['global_mean_abs_shap'] = mean_global[CIRCOS_TASKS].mean(axis=1)
        global_features = mean_global.sort_values('global_mean_abs_shap', ascending=False).head(2).index.tolist()
        used = set(global_features)
        for rank, feat in enumerate(global_features, start=1):
            records.append({'modal': modal_name, 'feature': feat, 'feature_type': 'global', 'assigned_task': 'Shared', 'rank': rank, 'mean_abs_shap': float(mean_global.loc[feat, 'global_mean_abs_shap'])})
        for task in CIRCOS_TASKS:
            task_ranked = mean_abs_by_task[task].sort_values(ascending=False)
            selected_feat = None
            for feat in task_ranked.index:
                if feat not in used:
                    selected_feat = feat
                    used.add(feat)
                    break
            if selected_feat is None:
                selected_feat = task_ranked.index[0]
            records.append({'modal': modal_name, 'feature': selected_feat, 'feature_type': 'specific', 'assigned_task': task, 'rank': 1, 'mean_abs_shap': float(mean_abs_by_task[task].loc[selected_feat])})
    table = pd.DataFrame(records)
    table.to_csv(CIRCOS_SOURCE_DIR / get_value('science.tables.result_213', 'result_213.csv'), index=False, encoding='utf-8-sig')
    return table
circos_feature_table_current = build_current_stage1_circos_feature_table()
display(circos_feature_table_current)

def prepare_stage1_circos_data_current(modal_name):
    
    circos_input_dir = _circos_input_dir(modal_name)
    x_visual_path = circos_input_dir / f"{modal_name}{get_value('science.tables.result_003', 'result_003.csv')}"
    pred_path = circos_input_dir / f"{modal_name}{get_value('science.tables.predictions_002', 'predictions_002.csv')}"
    if not x_visual_path.exists():
        raise FileNotFoundError(x_visual_path)
    if not pred_path.exists():
        raise FileNotFoundError(pred_path)
    X_visual = _circos_read_indexed(x_visual_path)
    pred_df = _circos_read_indexed(pred_path)
    shap_signed = {}
    shap_abs = {}
    for task in CIRCOS_TASKS:
        shap_signed[task] = _circos_read_indexed(_circos_signed_path(modal_name, task), compression='gzip')
        shap_abs[task] = _circos_read_indexed(_circos_abs_path(modal_name, task), compression='gzip')
        if task in ['PFS', 'OS']:
            shap_signed[task] = -shap_signed[task]
    common_samples = X_visual.index.intersection(pred_df.index)
    for task in CIRCOS_TASKS:
        common_samples = common_samples.intersection(shap_signed[task].index)
        common_samples = common_samples.intersection(shap_abs[task].index)
    X_visual = X_visual.loc[common_samples]
    pred_df = pred_df.loc[common_samples]
    for task in CIRCOS_TASKS:
        shap_signed[task] = shap_signed[task].loc[common_samples]
        shap_abs[task] = shap_abs[task].loc[common_samples]
    ft = circos_feature_table_current[circos_feature_table_current['modal'].astype(str).str.lower().eq(modal_name.lower())].copy()
    global_features = ft[ft['feature_type'] == 'global'].sort_values('rank')['feature'].astype(str).tolist()
    global_features = [f for f in global_features if f in X_visual.columns]
    specific_feature_map = {}
    for task in CIRCOS_TASKS:
        ss = ft[(ft['feature_type'] == 'specific') & (ft['assigned_task'].astype(str) == task)].copy()
        feat = ss.sort_values('rank')['feature'].astype(str).iloc[0] if len(ss) > 0 else None
        if feat not in X_visual.columns:
            print(f'[Warning] {modal_name} {task}: selected feature not available in X_visual: {feat}')
            feat = None
        specific_feature_map[task] = feat
    plot_features = []
    for i, feat in enumerate(global_features):
        plot_features.append({'name': feat, 'type': 'global', 'color': SHARED_COLORS[min(i, len(SHARED_COLORS) - 1)]})
    for task in CIRCOS_TASKS:
        feat = specific_feature_map.get(task)
        if feat is not None:
            plot_features.append({'name': feat, 'type': task, 'color': ENDPOINT_COLORS[task]})
    metadata_cols = [c for c in ['Response_true', 'PFS_status', 'OS_status'] if c in pred_df.columns]
    metadata = pred_df[metadata_cols].copy()
    for col in metadata.columns:
        metadata[col] = pd.to_numeric(metadata[col], errors='coerce').fillna(0).astype(int)
    metadata = metadata.rename(columns={'Response_true': 'Response', 'PFS_status': 'PFS event', 'OS_status': 'OS event'})
    result_list = []
    for task_idx, task in enumerate(CIRCOS_TASKS):
        df_task = pd.DataFrame(index=common_samples)
        if task == 'Response':
            true_raw = pd.to_numeric(pred_df['Response_true'], errors='coerce').values
            pred_raw = pd.to_numeric(pred_df['Response_pred'], errors='coerce').values
            df_task['True_Score'] = _circos_minmax_01(true_raw, jitter_binary=True, seed=42 + task_idx)
            df_task['Pred_Score'] = _circos_minmax_01(pred_raw, jitter_binary=False)
        elif task == 'PFS':
            true_raw = pd.to_numeric(pred_df['PFS_time'], errors='coerce').values
            pred_raw = -pd.to_numeric(pred_df['PFS_risk'], errors='coerce').values
            df_task['True_Score'] = _circos_minmax_01(true_raw)
            df_task['Pred_Score'] = _circos_minmax_01(pred_raw)
        else:
            true_raw = pd.to_numeric(pred_df['OS_time'], errors='coerce').values
            pred_raw = -pd.to_numeric(pred_df['OS_risk'], errors='coerce').values
            df_task['True_Score'] = _circos_minmax_01(true_raw)
            df_task['Pred_Score'] = _circos_minmax_01(pred_raw)
        shared_features = [f for f in global_features if f in shap_signed[task].columns]
        if shared_features:
            df_task['shared_shap_sum'] = shap_signed[task][shared_features].sum(axis=1)
        else:
            df_task['shared_shap_sum'] = 0.0
        specific_feat = specific_feature_map.get(task)
        if specific_feat is not None and specific_feat in shap_signed[task].columns:
            df_task['specific_shap_sum'] = shap_signed[task][specific_feat]
        else:
            df_task['specific_shap_sum'] = 0.0
        for f_info in plot_features:
            feat = f_info['name']
            if feat in X_visual.columns:
                df_task[feat] = X_visual[feat].values
        df_task = df_task.sort_values('Pred_Score')
        result_list.append(df_task)
        df_task.to_csv(CIRCOS_SOURCE_DIR / f"{modal_name}_{task}{get_value('science.tables.result_214', 'result_214.csv')}", encoding='utf-8-sig')
    print(f'[{modal_name}] aligned samples = {len(common_samples)} | input features = {X_visual.shape[1]}')
    print('  shared:', global_features)
    print('  endpoint-preferred:', specific_feature_map)
    return {'modal_name': modal_name, 'metadata': metadata, 'global_features': global_features, 'specific_feature_map': specific_feature_map, 'plot_features': plot_features, 'result_list': result_list}

def render_stage1_circos_current(circos_data, output_prefix, panel_label=None):
    modal_name = circos_data['modal_name']
    metadata = circos_data['metadata']
    plot_features = circos_data['plot_features']
    result_list = circos_data['result_list']
    sector_width = 10
    r_pad_ratio = 0.05
    sectors = {task: sector_width for task in CIRCOS_TASKS}
    circos = Circos(sectors, space=1.5)
    center_name = _circos_display_modal_name(modal_name)
    circos.text(center_name, size=FONT_CENTER, fontweight='bold', color=MODALITY_COLORS.get(modal_name, COLOR_TEXT))
    for index, sector in enumerate(circos.sectors):
        task = sector.name
        df_plot = result_list[index]
        num_patients = df_plot.shape[0]
        sector.text(task, r=101, size=FONT_SECTOR, fontweight='bold', color=ENDPOINT_COLORS[task])
        x = np.linspace(sector.start + r_pad_ratio, sector.end - r_pad_ratio, num_patients)
        out_pos = 100
        inner_pos = out_pos - CIRCOS_SIZE[0]
        track0 = sector.add_track((inner_pos, out_pos), r_pad_ratio=r_pad_ratio)
        track0.axis(fc='none', ec='#BEBEBE', lw=0.4)
        hm_raw = df_plot[['specific_shap_sum', 'shared_shap_sum']].abs().values.T
        row_min = hm_raw.min(axis=1, keepdims=True)
        row_max = hm_raw.max(axis=1, keepdims=True)
        hm_data = (hm_raw - row_min) / (row_max - row_min + 1e-08)
        hm_data = np.power(hm_data, 0.5)
        track0.heatmap(hm_data, cmap=_sector_heatmap_cmap(task))
        if index == 0:
            track0.yticks([inner_pos + 1.5, inner_pos + 4.5], ['Preferred SHAP', 'Shared SHAP'], side='left', label_size=FONT_LABEL_L2)
        out_pos = inner_pos
        inner_pos = out_pos - CIRCOS_SIZE[1]
        track1 = sector.add_track((inner_pos, out_pos), r_pad_ratio=r_pad_ratio)
        track1.axis(fc='none', ec='#BEBEBE', lw=0.5)
        track1.grid(lw=0.25, ls=':', color='#E6E6E6')
        bar_data = df_plot[['specific_shap_sum', 'shared_shap_sum']].values
        specific_color = ENDPOINT_COLORS[task]
        colors = [specific_color, '#7EC9C3']
        pos_sums = np.sum(np.where(bar_data > 0, bar_data, 0), axis=1)
        neg_sums = np.sum(np.where(bar_data < 0, bar_data, 0), axis=1)
        max_abs_stack = max(float(np.nanmax(pos_sums)), float(abs(np.nanmin(neg_sums))), 1e-06)
        vmin = -1.05 * max_abs_stack
        vmax = 1.05 * max_abs_stack
        for i in range(2):
            values = [bar_data[g][i] for g in range(num_patients)]
            bottoms_pos = [sum((bar_data[g][j] for j in range(i) if bar_data[g][j] > 0)) for g in range(num_patients)]
            bottoms_neg = [sum((bar_data[g][j] for j in range(i) if bar_data[g][j] < 0)) for g in range(num_patients)]
            bottoms = [bottoms_pos[g] if values[g] > 0 else bottoms_neg[g] for g in range(num_patients)]
            track1.bar(x, values, color=colors[i], width=(sector.end - sector.start) / num_patients, bottom=bottoms, vmin=vmin, vmax=vmax)
        if index == 0:
            track1.yticks([out_pos - CIRCOS_SIZE[1] / 2], ['Benefit-oriented\nSHAP'], side='left', label_size=FONT_LABEL_L1, line_kws=dict(lw=0))
        out_pos = inner_pos
        inner_pos = out_pos - CIRCOS_SIZE[2]
        track2 = sector.add_track((inner_pos, out_pos), r_pad_ratio=r_pad_ratio)
        track2.axis(fc='none', ec='#BEBEBE', lw=0.5)
        hm_meta = metadata.loc[df_plot.index].values.T
        track2.heatmap(hm_meta, cmap=_barcode_cmap())
        if index == 0 and metadata.shape[1] > 0:
            ytick_positions = np.linspace(inner_pos + 1, out_pos - 1, metadata.shape[1])
            track2.yticks(ytick_positions, metadata.columns[::-1], side='left', label_size=FONT_LABEL_L2)
        out_pos = inner_pos
        inner_pos = out_pos - CIRCOS_SIZE[3]
        track3 = sector.add_track((inner_pos, out_pos), r_pad_ratio=r_pad_ratio)
        track3.axis(fc='none', ec='#BEBEBE', lw=0.5)
        track3.scatter(x, df_plot['True_Score'].values, vmin=-0.1, vmax=1.1, color=OBSERVED_COLOR, s=13, alpha=0.98)
        track3.scatter(x, df_plot['Pred_Score'].values, vmin=-0.1, vmax=1.1, color=ENDPOINT_COLORS[task], s=13, alpha=0.98)
        if index == 0:
            track3.yticks([out_pos - CIRCOS_SIZE[3] / 2], ['Observed vs\nprojected'], side='left', label_size=FONT_LABEL_L1, line_kws=dict(lw=0))
        out_pos = inner_pos
        start = out_pos
        for f_info in plot_features:
            feat = f_info['name']
            f_type = f_info['type']
            f_color = f_info['color']
            if f_type == 'global' or f_type == task:
                if feat not in df_plot.columns:
                    start -= TRACK4_STEP
                    continue
                track4 = sector.add_track((start - TRACK4_STEP + 0.5, start), r_pad_ratio=r_pad_ratio)
                y_feat = pd.to_numeric(df_plot[feat], errors='coerce').values.astype(float)
                y_min = float(np.nanmin(y_feat))
                y_max = float(np.nanmax(y_feat))
                if y_min == y_max:
                    y_max += 1e-05
                y_vmax = y_max + (y_max - y_min) * 0.15
                track4.line(x, [y_min] * len(x), vmin=y_min, vmax=y_vmax, color='#D7D7D7', lw=0.5, ls='--', alpha=0.98)
                track4.line(x, y_feat, vmin=y_min, vmax=y_vmax, color=f_color, lw=1.25)
                if f_type == 'global' and index == 0 or f_type == task:
                    track4.yticks([start - TRACK4_STEP / 2], [_circos_clean_feature_name(feat, 18)], side='left', label_size=FONT_LABEL_L2, line_kws=dict(lw=0))
            start -= TRACK4_STEP
    fig = circos.plotfig()
    fig.set_size_inches(CIRCOS_CANVAS_SIZE)
    output_prefix = Path(output_prefix)
    fig.savefig(str(output_prefix) + '.png', dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(str(output_prefix) + '.pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(str(output_prefix) + '.svg', bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
single_modal_circos_outputs = {}
for modal_name in CIRCOS_MODALITIES:
    
    
    
    data = prepare_stage1_circos_data_current(modal_name)
    prefix = CIRCOS_OUT_DIR / f'{PANEL_LABELS[modal_name]}_Stage1_Circos_{modal_name}_Response_PFS_OS'
    render_stage1_circos_current(circos_data=data, output_prefix=prefix, panel_label=PANEL_LABELS[modal_name])
    single_modal_circos_outputs[modal_name] = data




In [ ]:
SUPP_OUT_DIR = OUT_DIR / 'Supplement'
SENS_OUT_DIR = SUPP_OUT_DIR / get_value('science.paths.directory_148', 'directory_148')
SENS_SOURCE_DIR = SOURCE_OUT_DIR / get_value('science.paths.directory_148', 'directory_148')
for d in [SUPP_OUT_DIR, SENS_OUT_DIR, SENS_SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def build_per_feature_normalized_modality_contribution():
    d = imp_mru[imp_mru['outcome_clean'].isin(ENDPOINT_ORDER)].copy()
    counts = feature_preference.groupby('modal_clean')['feature_clean'].nunique().rename('n_retained_features').reset_index()
    agg = d.groupby(['outcome_clean', 'modal_clean'], as_index=False).agg(total_mean_abs_shap=('mean_abs_shap_clean', 'sum')).merge(counts, on='modal_clean', how='left')
    agg['per_feature_mean_abs_shap'] = agg['total_mean_abs_shap'] / agg['n_retained_features'].replace(0, np.nan)
    agg['relative_per_feature_contribution'] = agg.groupby('outcome_clean')['per_feature_mean_abs_shap'].transform(lambda s: s / s.sum() if float(s.sum()) > 0 else np.nan)
    agg['relative_total_contribution'] = agg.groupby('outcome_clean')['total_mean_abs_shap'].transform(lambda s: s / s.sum() if float(s.sum()) > 0 else np.nan)
    agg.to_csv(SENS_SOURCE_DIR / get_value('science.tables.result_215', 'result_215.csv'), index=False, encoding='utf-8-sig')
    return agg
sxe_modality = build_per_feature_normalized_modality_contribution()

def plot_sxe_per_feature_normalized_modality_contribution():
    d = sxe_modality.copy()
    fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2), facecolor='white', sharey=True)
    x = np.arange(len(ENDPOINT_ORDER))
    width = 0.2
    for i, modal in enumerate(MODALITY_ORDER_MRU):
        ss = d[d['modal_clean'] == modal].set_index('outcome_clean').reindex(ENDPOINT_ORDER)
        axes[0].bar(x + (i - 1) * width, ss['relative_total_contribution'].values, width=width, label=modal, color=MODALITY_COLORS.get(modal, '#777777'), edgecolor='white', linewidth=0.6)
        axes[1].bar(x + (i - 1) * width, ss['relative_per_feature_contribution'].values, width=width, label=modal, color=MODALITY_COLORS.get(modal, '#777777'), edgecolor='white', linewidth=0.6)
    axes[0].set_title('Original modality contribution', fontsize=11.8, pad=8)
    axes[1].set_title('After per-feature normalization', fontsize=11.8, pad=8)
    for ax in axes:
        ax.set_xticks(x)
        ax.set_xticklabels(ENDPOINT_ORDER)
        ax.set_ylim(0, 0.8)
        ax.set_yticks(np.arange(0, 0.81, 0.2))
        ax.set_yticklabels([f'{int(v * 100)}%' for v in np.arange(0, 0.81, 0.2)])
        ax.set_ylabel('Relative modality contribution')
        clean_axes(ax, grid_axis='y')
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=MODALITY_COLORS[m], edgecolor='white') for m in MODALITY_ORDER_MRU]
    fig.legend(handles, MODALITY_ORDER_MRU, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=3, frameon=False, fontsize=10.0, title='Modality', title_fontsize=10.0)
    add_panel_header(fig, 'SxE', 'Modality-level attribution after accounting for unequal retained feature counts')
    fig.subplots_adjust(left=0.09, right=0.98, top=0.82, bottom=0.11, wspace=0.16)
    save_figure(fig, SENS_OUT_DIR / get_value('science.paths.directory_149', 'directory_149'))
    plt.show()
    return d


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
_required_label_cols = ['Response_true', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']
_missing_label_cols = [c for c in _required_label_cols if c not in pred_mru.columns]
if _missing_label_cols:
    raise ValueError(get_value('science.messages.status_028', 'input is missing required observed-label columns: ') + ', '.join(_missing_label_cols) + f'\nExisting columns: {list(pred_mru.columns)}')
y_sens = pred_mru[_required_label_cols].rename(columns={'Response_true': 'Response'}).reindex(X_mru.index).copy()
for c in ['Response', 'PFS_time', 'PFS_status', 'OS_time', 'OS_status']:
    y_sens[c] = pd.to_numeric(y_sens[c], errors='coerce')
print(get_value('science.messages.status_029', '\nSxF sensitivity labels reconstructed from input'))
print('  X_mru:', X_mru.shape)
print('  y_sens:', y_sens.shape)
print('  Response non-missing:', int(y_sens['Response'].notna().sum()))
print('  PFS non-missing:', int((y_sens['PFS_time'].notna() & y_sens['PFS_status'].notna()).sum()))
print('  OS non-missing:', int((y_sens['OS_time'].notna() & y_sens['OS_status'].notna()).sum()))

def _normalize_importance(vec):
    s = pd.Series(vec, index=X_mru.columns, dtype=float).abs().replace([np.inf, -np.inf], np.nan).fillna(0.0)
    tot = float(s.sum())
    if tot <= 0:
        return s
    return s / tot

def _fit_response_alternative_models():
    X = X_mru.copy()
    y = pd.to_numeric(y_sens['Response'], errors='coerce').astype(float)
    mask = y.notna()
    X = X.loc[mask]
    y = y.loc[mask].astype(int)
    results = []
    pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', LogisticRegression(penalty='l2', C=1.0, solver='liblinear', max_iter=5000, random_state=42))])
    pipe.fit(X, y)
    coef = pipe.named_steps['model'].coef_.ravel()
    results.append({'task': 'Response', 'model_family': 'Logistic', 'importance': _normalize_importance(coef)})
    tree_models = [('RandomForest', RandomForestClassifier(n_estimators=1000, random_state=42, class_weight='balanced', min_samples_leaf=2, n_jobs=-1)), ('ExtraTrees', ExtraTreesClassifier(n_estimators=1000, random_state=42, class_weight='balanced', min_samples_leaf=2, n_jobs=-1)), ('GradientBoosting', GradientBoostingClassifier(random_state=42))]
    for name, model in tree_models:
        pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', model)])
        pipe.fit(X, y)
        imp = getattr(pipe.named_steps['model'], 'feature_importances_', None)
        if imp is None:
            p = permutation_importance(pipe, X, y, scoring='roc_auc', n_repeats=20, random_state=42, n_jobs=-1)
            imp = p.importances_mean
        results.append({'task': 'Response', 'model_family': name, 'importance': _normalize_importance(imp)})
    return results

def _fit_lifelines_cox_one_task(task, X):
    from lifelines import CoxPHFitter
    time_col = f'{task}_time'
    event_col = f'{task}_status'
    time = pd.to_numeric(y_sens[time_col], errors='coerce')
    event = pd.to_numeric(y_sens[event_col], errors='coerce')
    mask = time.notna() & event.notna()
    imp = SimpleImputer(strategy='median')
    X_imp = pd.DataFrame(imp.fit_transform(X.loc[mask]), index=X.loc[mask].index, columns=X.columns)
    scaler = StandardScaler()
    X_std = pd.DataFrame(scaler.fit_transform(X_imp), index=X_imp.index, columns=X_imp.columns)
    df = X_std.copy()
    df[time_col] = time.loc[mask].astype(float).values
    df[event_col] = event.loc[mask].astype(int).values
    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(df, duration_col=time_col, event_col=event_col)
    coef = cph.params_.reindex(X.columns).fillna(0.0).values
    return {'task': task, 'model_family': 'CoxPH', 'importance': _normalize_importance(coef)}

def _fit_survival_alternative_models():
    results = []
    X = X_mru.copy()
    sksurv_available = False
    try:
        import sklearn
        import sksurv
        from sksurv.util import Surv
        from sksurv.linear_model import CoxPHSurvivalAnalysis
        from sksurv.ensemble import GradientBoostingSurvivalAnalysis
        sksurv_available = True
        print(f'[SxF] scikit-learn={sklearn.__version__} | scikit-survival={sksurv.__version__}')
    except Exception as e:
        print('[Info] scikit-survival unavailable; survival sensitivity will use lifelines CoxPH only.')
        print('Reason:', repr(e))
    for task in ['PFS', 'OS']:
        time_col = f'{task}_time'
        event_col = f'{task}_status'
        time = pd.to_numeric(y_sens[time_col], errors='coerce')
        event = pd.to_numeric(y_sens[event_col], errors='coerce')
        mask = time.notna() & event.notna()
        X_task = X.loc[mask].copy()
        cox_done = False
        if sksurv_available:
            y_struct = Surv.from_arrays(event=event.loc[mask].astype(bool).values, time=time.loc[mask].astype(float).values)
            try:
                cox_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', CoxPHSurvivalAnalysis(alpha=0.001))])
                cox_pipe.fit(X_task, y_struct)
                coef = cox_pipe.named_steps['model'].coef_.ravel()
                results.append({'task': task, 'model_family': 'CoxPH', 'importance': _normalize_importance(coef)})
                cox_done = True
                print(f'[SxF] {task} CoxPH completed.')
            except Exception as e:
                print(f'[Warning] scikit-survival CoxPH failed for {task}; trying lifelines CoxPH for this task only.')
                print('Reason:', repr(e))
            try:
                imputer = SimpleImputer(strategy='median')
                X_imp = imputer.fit_transform(X_task)
                survival_gb = GradientBoostingSurvivalAnalysis(loss='coxph', learning_rate=0.03, n_estimators=300, subsample=0.8, min_samples_split=10, min_samples_leaf=5, max_depth=2, random_state=42)
                survival_gb.fit(X_imp, y_struct)
                gb_importance = np.asarray(survival_gb.feature_importances_, dtype=float)
                if gb_importance.ndim != 1 or gb_importance.shape[0] != X.shape[1]:
                    raise RuntimeError(f'Unexpected SurvivalGB importance shape: {gb_importance.shape}; expected ({X.shape[1]},)')
                results.append({'task': task, 'model_family': 'SurvivalGB', 'importance': _normalize_importance(gb_importance)})
                print(f'[SxF] {task} SurvivalGB completed.')
            except Exception as e:
                print(f'[Warning] SurvivalGB sensitivity failed for {task}; CoxPH result is retained.')
                print('Reason:', repr(e))
        if not cox_done:
            try:
                results.append(_fit_lifelines_cox_one_task(task=task, X=X))
                cox_done = True
                print(f'[SxF] {task} CoxPH completed with lifelines fallback.')
            except Exception as e:
                print(f'[Warning] CoxPH sensitivity failed for {task} in both available backends.')
                print('Reason:', repr(e))
    keys = [(item['task'], item['model_family']) for item in results]
    if len(keys) != len(set(keys)):
        dup = pd.Series(keys)[pd.Series(keys).duplicated(keep=False)].tolist()
        raise RuntimeError('Duplicate survival model-task combinations detected: ' + str(dup))
    return results

def build_alternative_model_importance_table(top_n=15, top_k_support=10):
    dnn_ref = imp_mru[imp_mru['outcome_clean'].isin(ENDPOINT_ORDER)].groupby(['feature_clean', 'modal_clean'], as_index=False).agg(dnn_mean_abs_shap=('mean_abs_shap_clean', 'mean'))
    dnn_ref['dnn_norm_importance'] = dnn_ref['dnn_mean_abs_shap'] / dnn_ref['dnn_mean_abs_shap'].sum()
    alt_results = []
    alt_results.extend(_fit_response_alternative_models())
    alt_results.extend(_fit_survival_alternative_models())
    model_task_keys = [(item['task'], item['model_family']) for item in alt_results]
    if len(model_task_keys) != len(set(model_task_keys)):
        raise RuntimeError('SxF generated duplicate model-task combinations: ' + str(model_task_keys))
    print('\n[SxF] Unique model-task combinations:', model_task_keys)
    if len(alt_results) == 0:
        raise RuntimeError('No alternative-model importance results were generated. Please install either scikit-survival or lifelines for survival sensitivity.')
    long_rows = []
    for item in alt_results:
        task = item['task']
        model_family = item['model_family']
        imp = item['importance'].reindex(X_mru.columns).fillna(0.0)
        rank = imp.rank(ascending=False, method='min')
        support = (rank <= top_k_support).astype(int)
        for feat in X_mru.columns:
            modal = feature_preference.set_index('feature_clean').reindex([feat])['modal_clean'].iloc[0] if feat in set(feature_preference['feature_clean']) else np.nan
            long_rows.append({'task': task, 'model_family': model_family, 'feature_clean': feat, 'modal_clean': modal, 'importance_norm': float(imp.loc[feat]), 'rank': float(rank.loc[feat]), 'topk_support': int(support.loc[feat])})
    alt_long = pd.DataFrame(long_rows)
    expected_rows = len(model_task_keys) * X_mru.shape[1]
    if len(alt_long) != expected_rows:
        raise RuntimeError(f'SxF row-count mismatch: got {len(alt_long)}, expected {expected_rows} = {len(model_task_keys)} model-task combinations × {X_mru.shape[1]} MRU features.')
    print(f'[SxF] Long table validated: {len(alt_long)} rows = {len(model_task_keys)} unique model-task combinations × {X_mru.shape[1]} features.')
    display_features = dnn_ref.sort_values('dnn_norm_importance', ascending=False).head(top_n)['feature_clean'].tolist()
    support_summary = alt_long[alt_long['feature_clean'].isin(display_features)].groupby(['feature_clean', 'modal_clean'], as_index=False).agg(support_count=('topk_support', 'sum'), mean_alt_importance=('importance_norm', 'mean'), median_alt_rank=('rank', 'median'), n_model_tasks=('topk_support', 'count')).merge(dnn_ref[['feature_clean', 'dnn_norm_importance']], on='feature_clean', how='left')
    support_summary['support_fraction'] = support_summary['support_count'] / support_summary['n_model_tasks'].replace(0, np.nan)
    alt_long.to_csv(SENS_SOURCE_DIR / get_value('science.tables.attribution_032', 'attribution_032.csv'), index=False, encoding='utf-8-sig')
    support_summary.to_csv(SENS_SOURCE_DIR / get_value('science.tables.metrics_076', 'metrics_076.csv'), index=False, encoding='utf-8-sig')
    return (alt_long, support_summary, display_features)
sxf_long, sxf_support, sxf_display_features = build_alternative_model_importance_table()

def plot_sxf_alternative_model_support():
    display_features = sxf_display_features.copy()
    disp = sxf_long[sxf_long['feature_clean'].isin(display_features)].copy()
    model_order = []
    for task in ['Response', 'PFS', 'OS']:
        cols = sorted(disp.loc[disp['task'] == task, 'model_family'].unique())
        for c in cols:
            model_order.append((task, c))
    disp['model_task'] = disp['task'] + ' | ' + disp['model_family']
    col_order = [f'{t} | {m}' for t, m in model_order]
    heat = disp.pivot_table(index='feature_clean', columns='model_task', values='importance_norm', aggfunc='mean', fill_value=0.0).reindex(columns=col_order, fill_value=0.0)
    feature_order = sxf_support.sort_values(['support_fraction', 'dnn_norm_importance'], ascending=False)['feature_clean'].tolist()
    feature_order = [f for f in feature_order if f in heat.index]
    heat = heat.reindex(feature_order)
    support_map = sxf_support.set_index('feature_clean').reindex(feature_order)
    fig = plt.figure(figsize=(11.4, max(5.4, 0.34 * len(feature_order) + 1.8)), facecolor='white')
    gs = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[1.0, 0.16], wspace=0.06)
    ax = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1], sharey=ax)
    im = ax.imshow(heat.values, aspect='auto', interpolation='nearest', cmap='YlGnBu')
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns, rotation=45, ha='right', fontsize=9.7)
    ax.set_yticks(np.arange(len(feature_order)))
    ax.set_yticklabels([clean_feature_label(f, 28) for f in feature_order], fontsize=9.7)
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    modal_map = feature_preference.drop_duplicates('feature_clean').set_index('feature_clean')['modal_clean'].to_dict()
    for i, f in enumerate(feature_order):
        ax.scatter(-0.7, i, s=46, marker='s', facecolor=MODALITY_COLORS.get(modal_map.get(f, ''), '#AAAAAA'), edgecolor='white', linewidth=0.5, clip_on=False, zorder=5)
    ax2.barh(np.arange(len(feature_order)), support_map['support_fraction'].values, color='#7A9CC6', edgecolor='white', linewidth=0.5)
    ax2.set_xlim(0, 1.0)
    ax2.set_xticks([0, 0.5, 1.0])
    ax2.set_xticklabels(['0%', '50%', '100%'], fontsize=9.7)
    ax2.set_xlabel('Top-k\nsupport', fontsize=9.7)
    ax2.tick_params(axis='y', left=False, labelleft=False)
    clean_axes(ax2, grid_axis='x')
    cbar = fig.colorbar(im, ax=[ax, ax2], fraction=0.02, pad=0.012)
    cbar.set_label('Normalized feature importance', fontsize=9.7)
    cbar.ax.tick_params(labelsize=6.5)
    cbar.outline.set_visible(False)
    add_panel_header(fig, 'SxF', 'Sensitivity of core MRU determinants across alternative model families')
    fig.subplots_adjust(left=0.26, right=0.95, top=0.88, bottom=0.12)
    save_figure(fig, SENS_OUT_DIR / get_value('science.paths.directory_150', 'directory_150'))
    plt.show()
    return (heat, support_map)


In [ ]:
sxe_output = plot_sxe_per_feature_normalized_modality_contribution()
sxf_output = plot_sxf_alternative_model_support()





In [ ]:
SXG_OUT_DIR = SUPP_OUT_DIR / get_value('science.paths.directory_151', 'directory_151')
SXG_SOURCE_DIR = SOURCE_OUT_DIR / get_value('science.paths.directory_151', 'directory_151')
for d in [SXG_OUT_DIR, SXG_SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def build_sxg_full_stability_table():
    if 'stage1_stability_endpoint' not in globals():
        raise RuntimeError('stage1_stability_endpoint is not available. Run the Figure 4F bootstrap-stability cell first.')
    d = stage1_stability_endpoint.copy()
    nfeat_records = []
    for modal in MODALITY_ORDER_STAGE1:
        for task in ENDPOINT_ORDER:
            p = _stage1_shap_abs_path(modal, task)
            if not p.exists():
                continue
            S = read_indexed_csv(p)
            nfeat_records.append({'modal_clean': modal, 'outcome_clean': task, 'n_stage1_features': int(S.shape[1])})
    nfeat_df = pd.DataFrame(nfeat_records)
    d = d.merge(nfeat_df, on=['modal_clean', 'outcome_clean'], how='left')
    denom = (d['n_stage1_features'] - 1).replace(0, np.nan)
    d['median_rank_quality'] = (1.0 - (d['bootstrap_median_rank'] - 1.0) / denom).clip(0, 1)
    pattern_map = golden[['modal_clean', 'feature_clean', 'pattern']].drop_duplicates()
    d = d.merge(pattern_map, on=['modal_clean', 'feature_clean'], how='left')
    d.to_csv(SXG_SOURCE_DIR / get_value('science.tables.result_216', 'result_216.csv'), index=False, encoding='utf-8-sig')
    return d
sxg_stability = build_sxg_full_stability_table()

def plot_sxg_full_stage1_bootstrap_stability():
    d = sxg_stability.copy()
    modalities = [m for m in MODALITY_ORDER_STAGE1 if m in d['modal_clean'].unique()]
    fig, axes = plt.subplots(1, len(modalities), figsize=(12.8, 7.6), facecolor='white', sharex=False, sharey=False)
    if len(modalities) == 1:
        axes = [axes]
    cmap = mpl.cm.Blues
    norm = mpl.colors.Normalize(vmin=0, vmax=1)
    all_source = []
    for ax, modal in zip(axes, modalities):
        ss = d[d['modal_clean'] == modal].copy()
        row_summary = ss.groupby(['feature_clean', 'pattern'], as_index=False).agg(mean_freq=('bootstrap_topk_frequency', 'mean'), max_freq=('bootstrap_topk_frequency', 'max'), n_original_selected=('originally_selected', 'sum'))
        row_summary = row_summary.sort_values(['n_original_selected', 'mean_freq', 'max_freq'], ascending=[False, False, False])
        features = row_summary['feature_clean'].tolist()
        y_map = {f: i for i, f in enumerate(features)}
        x_map = {t: i for i, t in enumerate(ENDPOINT_ORDER)}
        for x in range(len(ENDPOINT_ORDER)):
            ax.axvline(x, color='#EFEFEF', linewidth=0.6, zorder=0)
        for _, r in ss.iterrows():
            feat = r['feature_clean']
            task = r['outcome_clean']
            if feat not in y_map or task not in x_map:
                continue
            freq = float(r['bootstrap_topk_frequency'])
            rank_quality = float(r['median_rank_quality']) if np.isfinite(r['median_rank_quality']) else 0.0
            selected = int(r['originally_selected']) == 1
            size = 20 + 260 * max(freq, 0) ** 0.85
            ax.scatter(x_map[task], y_map[feat], s=size, c=[rank_quality], cmap=cmap, norm=norm, edgecolor='#222222' if selected else 'white', linewidth=0.9 if selected else 0.45, alpha=0.98, zorder=3)
        for f in features:
            ax.scatter(-0.55, y_map[f], s=39, marker='s', facecolor=MODALITY_COLORS.get(modal, '#999999'), edgecolor='white', linewidth=0.45, clip_on=False, zorder=4)
        ax.set_xticks(np.arange(len(ENDPOINT_ORDER)))
        ax.set_xticklabels(ENDPOINT_ORDER, rotation=35, ha='right', fontsize=10.0)
        ax.set_yticks(np.arange(len(features)))
        ax.set_yticklabels([clean_feature_label(f, 26) for f in features], fontsize=9.2)
        ax.set_ylim(len(features) - 0.5, -0.5)
        ax.set_xlim(-0.78, len(ENDPOINT_ORDER) - 0.45)
        ax.tick_params(length=0)
        for sp in ax.spines.values():
            sp.set_visible(False)
        ax.set_title(modal, fontsize=11.8, fontweight='bold', color=MODALITY_COLORS.get(modal, COLOR_TEXT), pad=8)
        med = float(ss.loc[ss['originally_selected'] == 1, 'bootstrap_topk_frequency'].median())
        ax.text(0.5, 1.01, f'median retention = {med:.0%}', transform=ax.transAxes, ha='center', va='bottom', fontsize=9.2, color=COLOR_MUTED)
    size_levels = [0.25, 0.5, 0.75, 1.0]
    handles = [plt.scatter([], [], s=20 + 260 * v ** 0.85, facecolor='#8BB9DD', edgecolor='white', linewidth=0.5) for v in size_levels]
    fig.legend(handles, [f'{int(v * 100)}%' for v in size_levels], title='Bootstrap Top-k\nretention', loc='upper center', bbox_to_anchor=(0.55, 0.905), ncol=4, frameon=False, fontsize=9.7, title_fontsize=9.7)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cax = fig.add_axes([0.915, 0.19, 0.012, 0.56])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label('Bootstrap median-rank quality\n(1 = highest rank)', fontsize=9.7)
    cbar.ax.tick_params(labelsize=6.4)
    cbar.outline.set_visible(False)
    add_panel_header(fig, 'SxG', 'Complete feature-level robustness of Stage-1 SHAP prioritization')
    fig.subplots_adjust(left=0.1, right=0.885, top=0.79, bottom=0.09, wspace=0.58)
    save_figure(fig, SXG_OUT_DIR / get_value('science.paths.directory_152', 'directory_152'))
    plt.show()
    return d
sxg_output = plot_sxg_full_stage1_bootstrap_stability()


In [ ]:
FULL_BEESWARM_OUT_DIR = SUPP_OUT_DIR / get_value('science.paths.directory_153', 'directory_153')
FULL_BEESWARM_SOURCE_DIR = SOURCE_OUT_DIR / get_value('science.paths.directory_153', 'directory_153')
for d in [FULL_BEESWARM_OUT_DIR, FULL_BEESWARM_SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def _feature_value_percentile(v):
    s = pd.Series(v, dtype=float)
    return s.rank(method='average', pct=True).values

def plot_full_mru_beeswarm_triptych():
    features = feature_preference.sort_values('overall_relative_importance', ascending=False)['feature_clean'].tolist()
    features = [f for f in features if f in X_mru.columns and all((f in shap_benefit[t].columns for t in ENDPOINT_ORDER))]
    features = list(dict.fromkeys(features))
    if len(features) == 0:
        raise ValueError('No common MRU features available for SxH–J.')
    plot_features = features[::-1]
    fig, axes = plt.subplots(1, 3, figsize=(14.2, max(10.5, 0.285 * len(plot_features) + 1.8)), facecolor='white', sharey=True)
    source_rows = []
    modal_map = feature_preference[['feature_clean', 'modal_clean']].drop_duplicates('feature_clean').set_index('feature_clean')['modal_clean'].to_dict()
    for ai, (ax, task) in enumerate(zip(axes, ENDPOINT_ORDER)):
        rng = np.random.default_rng(1000 + ai)
        for yi, feat in enumerate(plot_features):
            common = shap_benefit[task].index.intersection(X_mru.index)
            shap_vals = pd.to_numeric(shap_benefit[task].loc[common, feat], errors='coerce').values.astype(float)
            feat_vals = pd.to_numeric(X_mru.loc[common, feat], errors='coerce').values.astype(float)
            mask = np.isfinite(shap_vals) & np.isfinite(feat_vals)
            shap_vals = shap_vals[mask]
            feat_vals = feat_vals[mask]
            sample_ids = np.asarray(common)[mask]
            if len(shap_vals) == 0:
                continue
            value_pct = _feature_value_percentile(feat_vals)
            jitter = np.zeros(len(shap_vals), dtype=float)
            n_bins = 22
            lo = float(np.nanmin(shap_vals))
            hi = float(np.nanmax(shap_vals))
            if hi <= lo:
                bins = np.array([lo - 1e-06, hi + 1e-06])
            else:
                bins = np.linspace(lo, hi + 1e-12, n_bins + 1)
            bin_id = np.digitize(shap_vals, bins) - 1
            for b in np.unique(bin_id):
                idx = np.where(bin_id == b)[0]
                if len(idx) <= 1:
                    continue
                offsets = np.arange(len(idx), dtype=float)
                offsets = offsets - offsets.mean()
                max_abs = np.max(np.abs(offsets))
                if max_abs > 0:
                    offsets = offsets / max_abs * 0.18
                idx_sorted = idx[np.argsort(value_pct[idx])]
                jitter[idx_sorted] = offsets
            ax.scatter(shap_vals, yi + jitter, c=value_pct, cmap=FEATURE_VALUE_CMAP, vmin=0, vmax=1, s=17, alpha=0.92, edgecolors='none', rasterized=True, zorder=3)
            for sid, sv, fv, vp in zip(sample_ids, shap_vals, feat_vals, value_pct):
                source_rows.append({'sample': sid, 'outcome': task, 'feature': feat, 'feature_value': float(fv), 'feature_value_percentile': float(vp), 'benefit_oriented_shap': float(sv)})
        ax.axvline(0, color='#808080', linewidth=0.8, linestyle=(0, (3, 3)), zorder=1)
        ax.set_title(task, fontsize=11.0, fontweight='bold', color=ENDPOINT_COLORS[task], pad=4)
        ax.set_xlabel('Benefit-oriented SHAP value')
        ax.set_ylabel('')
        ax.set_yticks(np.arange(len(plot_features)))
        if ai == 0:
            ax.set_yticklabels([clean_feature_label(f, 30) for f in plot_features], fontsize=8.3)
            for lab in ax.get_yticklabels():
                lab.set_rotation(45)
                lab.set_rotation_mode('anchor')
                lab.set_ha('right')
                lab.set_va('center')
            xlim = ax.get_xlim()
            x_strip = xlim[0] - 0.055 * (xlim[1] - xlim[0])
            for yi, feat in enumerate(plot_features):
                ax.scatter(x_strip, yi, s=30, marker='s', facecolor=MODALITY_COLORS.get(modal_map.get(feat, ''), '#AAAAAA'), edgecolor='white', linewidth=0.35, clip_on=False, zorder=5)
            ax.set_xlim(xlim)
        else:
            ax.tick_params(axis='y', left=False, labelleft=False)
        clean_axes(ax, grid_axis='x')
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0, vmax=1), cmap=FEATURE_VALUE_CMAP)
    sm.set_array([])
    cax = fig.add_axes([0.93, 0.18, 0.012, 0.6])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(['Low', 'High'])
    cbar.set_label('Feature value', fontsize=9.0)
    cbar.ax.tick_params(labelsize=8.0)
    cbar.outline.set_visible(False)
    modality_handles = [Line2D([0], [0], marker='s', linestyle='', markerfacecolor=MODALITY_COLORS[m], markeredgecolor='white', markersize=6.5, label=m) for m in MODALITY_ORDER_MRU]
    fig.legend(handles=modality_handles, title='Modality', loc='upper center', bbox_to_anchor=(0.62, 0.985), ncol=3, frameon=False, fontsize=8.5, title_fontsize=8.8)
    source_df = pd.DataFrame(source_rows)
    source_df.to_csv(FULL_BEESWARM_SOURCE_DIR / get_value('science.tables.result_217', 'result_217.csv'), index=False, encoding='utf-8-sig')
    fig.subplots_adjust(left=0.25, right=0.91, top=0.94, bottom=0.07, wspace=0.16)
    save_figure(fig, FULL_BEESWARM_OUT_DIR / get_value('science.paths.directory_154', 'directory_154'))
    plt.show()
    return source_df
sxh_sxi_sxj_triptych = plot_full_mru_beeswarm_triptych()




In [ ]:
SXK_OUT_DIR = SUPP_OUT_DIR / get_value('science.paths.directory_155', 'directory_155')
SXK_SOURCE_DIR = SOURCE_OUT_DIR / get_value('science.paths.directory_155', 'directory_155')
for d in [SXK_OUT_DIR, SXK_SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def build_sxk_rank_shift_table():
    d = determinant_long[determinant_long['outcome_clean'].isin(ENDPOINT_ORDER)].copy()
    d['endpoint_rank'] = d.groupby('outcome_clean')['mean_abs_shap_clean'].rank(method='min', ascending=False)
    denom = d.groupby('outcome_clean')['mean_abs_shap_clean'].transform('sum').replace(0, np.nan)
    d['relative_importance'] = d['mean_abs_shap_clean'] / denom
    rank_wide = d.pivot_table(index=['feature_clean', 'modal_clean'], columns='outcome_clean', values='endpoint_rank', aggfunc='first').reset_index()
    imp_wide = d.pivot_table(index=['feature_clean', 'modal_clean'], columns='outcome_clean', values='relative_importance', aggfunc='first').reset_index()
    for task in ENDPOINT_ORDER:
        if task not in rank_wide.columns:
            rank_wide[task] = np.nan
        if task not in imp_wide.columns:
            imp_wide[task] = np.nan
    rank_wide['rank_min'] = rank_wide[ENDPOINT_ORDER].min(axis=1)
    rank_wide['rank_max'] = rank_wide[ENDPOINT_ORDER].max(axis=1)
    rank_wide['rank_range'] = rank_wide['rank_max'] - rank_wide['rank_min']
    overall_imp = d.groupby(['feature_clean', 'modal_clean'], as_index=False).agg(overall_mean_abs_shap=('mean_abs_shap_clean', 'mean'))
    rank_wide = rank_wide.merge(overall_imp, on=['feature_clean', 'modal_clean'], how='left')
    rank_wide = rank_wide.sort_values(['overall_mean_abs_shap', 'rank_range'], ascending=[False, False]).reset_index(drop=True)
    long_out = d[['feature_clean', 'modal_clean', 'outcome_clean', 'mean_abs_shap_clean', 'relative_importance', 'endpoint_rank']].copy()
    long_out.to_csv(SXK_SOURCE_DIR / get_value('science.tables.result_218', 'result_218.csv'), index=False, encoding='utf-8-sig')
    rank_wide.to_csv(SXK_SOURCE_DIR / get_value('science.tables.result_219', 'result_219.csv'), index=False, encoding='utf-8-sig')
    imp_wide.to_csv(SXK_SOURCE_DIR / get_value('science.tables.attribution_033', 'attribution_033.csv'), index=False, encoding='utf-8-sig')
    return (long_out, rank_wide, imp_wide)
sxk_long, sxk_rank, sxk_imp = build_sxk_rank_shift_table()

def plot_sxk_rank_shift():
    rank_df = sxk_rank.copy()
    imp_df = sxk_imp.set_index(['feature_clean', 'modal_clean']).reindex(pd.MultiIndex.from_frame(rank_df[['feature_clean', 'modal_clean']])).reset_index()
    features = rank_df['feature_clean'].tolist()
    modals = rank_df['modal_clean'].tolist()
    rank_matrix = rank_df[ENDPOINT_ORDER].values.astype(float)
    imp_matrix = imp_df[ENDPOINT_ORDER].values.astype(float)
    imp_scaled = imp_matrix.copy()
    for j in range(imp_scaled.shape[1]):
        col = imp_scaled[:, j]
        finite = np.isfinite(col)
        if finite.any():
            cmin = np.nanmin(col[finite])
            cmax = np.nanmax(col[finite])
            if cmax > cmin:
                imp_scaled[finite, j] = (col[finite] - cmin) / (cmax - cmin)
            else:
                imp_scaled[finite, j] = 0.5
    fig = plt.figure(figsize=(11.6, max(10.0, 0.29 * len(features) + 1.9)), facecolor='white')
    gs = fig.add_gridspec(nrows=1, ncols=3, width_ratios=[0.05, 1.0, 0.32], wspace=0.12)
    ax_strip = fig.add_subplot(gs[0, 0])
    ax = fig.add_subplot(gs[0, 1])
    ax_range = fig.add_subplot(gs[0, 2], sharey=ax)
    im = ax.imshow(imp_scaled, aspect='auto', interpolation='nearest', cmap='YlOrRd', vmin=0, vmax=1)
    for i in range(len(features)):
        for j in range(len(ENDPOINT_ORDER)):
            r = rank_matrix[i, j]
            if np.isfinite(r):
                val = imp_scaled[i, j]
                txt_color = 'white' if val >= 0.62 else COLOR_TEXT
                ax.text(j, i, f'{int(r)}', ha='center', va='center', fontsize=8.8, color=txt_color, fontweight='bold' if r <= 10 else 'normal')
    ax.set_xticks(np.arange(len(ENDPOINT_ORDER)))
    ax.set_xticklabels(ENDPOINT_ORDER, fontsize=10.8, fontweight='bold')
    ax.set_yticks(np.arange(len(features)))
    ax.set_yticklabels([clean_feature_label(f, 40) for f in features], fontsize=9.2)
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    strip_rgba = np.array([mpl.colors.to_rgba(MODALITY_COLORS.get(m, '#AAAAAA')) for m in modals]).reshape(len(modals), 1, 4)
    ax_strip.imshow(strip_rgba, aspect='auto', interpolation='nearest')
    ax_strip.set_xticks([])
    ax_strip.set_yticks([])
    for sp in ax_strip.spines.values():
        sp.set_visible(False)
    y = np.arange(len(features))
    ax_range.barh(y, rank_df['rank_range'].values, height=0.72, color=[MODALITY_COLORS.get(m, '#999999') for m in modals], edgecolor='white', linewidth=0.4, alpha=0.98)
    ax_range.set_xlabel('Rank range\n(max − min)', fontsize=9.7)
    ax_range.tick_params(axis='x', labelsize=6.2)
    ax_range.tick_params(axis='y', left=False, labelleft=False)
    clean_axes(ax_range, grid_axis='x')
    ax.set_ylim(len(features) - 0.5, -0.5)
    ax_range.set_ylim(len(features) - 0.5, -0.5)
    cax = fig.add_axes([0.915, 0.18, 0.013, 0.58])
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label('Within-endpoint relative\nimportance (scaled)', fontsize=9.7)
    cbar.ax.tick_params(labelsize=6.2)
    cbar.outline.set_visible(False)
    modality_handles = [Line2D([0], [0], marker='s', linestyle='', markerfacecolor=MODALITY_COLORS[m], markeredgecolor='white', markersize=7.0, label=m) for m in MODALITY_ORDER_MRU]
    fig.legend(handles=modality_handles, title='Modality', loc='upper center', bbox_to_anchor=(0.7, 0.925), ncol=3, frameon=False, fontsize=9.7, title_fontsize=9.7)
    add_panel_header(fig, 'SxK', 'Cross-endpoint rank shifts of all MRU determinants')
    fig.subplots_adjust(left=0.32, right=0.885, top=0.86, bottom=0.06)
    save_figure(fig, SXK_OUT_DIR / get_value('science.paths.directory_156', 'directory_156'))
    plt.show()
    return rank_df
sxk_output = plot_sxk_rank_shift()


In [ ]:
FULL_HEATMAP_OUT_DIR = SUPP_OUT_DIR / get_value('science.paths.directory_157', 'directory_157')
FULL_HEATMAP_SOURCE_DIR = SOURCE_OUT_DIR / get_value('science.paths.directory_157', 'directory_157')
for d in [FULL_HEATMAP_OUT_DIR, FULL_HEATMAP_SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
FULL_HEATMAP_PANEL = {'Response': 'SxL', 'PFS': 'SxM', 'OS': 'SxN'}

def _row_scale_full_heatmap(M, quantile=0.95):
    M = M.astype(float).copy()
    out = M.copy()
    denom = {}
    for idx in M.index:
        vals = M.loc[idx].values.astype(float)
        finite = np.abs(vals[np.isfinite(vals)])
        if len(finite) == 0:
            d = 1.0
        else:
            d = float(np.quantile(finite, quantile))
            if not np.isfinite(d) or d <= 1e-12:
                d = float(np.nanmax(finite)) if len(finite) else 1.0
            if not np.isfinite(d) or d <= 1e-12:
                d = 1.0
        denom[idx] = d
        out.loc[idx] = np.clip(vals / d, -1, 1)
    return (out, pd.Series(denom))

def _get_full_patient_order(task):
    common = shap_benefit[task].index.intersection(X_mru.index)
    ordered = [p for p in PATIENT_ORDER if p in common]
    if len(ordered) == 0:
        ordered = list(common)
    return ordered

def _get_full_feature_order(task):
    d = determinant_long[determinant_long['outcome_clean'] == task].sort_values('mean_abs_shap_clean', ascending=False)
    features = [f for f in d['feature_clean'].tolist() if f in shap_benefit[task].columns]
    return list(dict.fromkeys(features))

def plot_full_patient_shap_heatmap(task):
    if task not in ENDPOINT_ORDER:
        raise ValueError(task)
    panel = FULL_HEATMAP_PANEL[task]
    patients = _get_full_patient_order(task)
    features = _get_full_feature_order(task)
    if len(patients) == 0:
        raise ValueError(f'No patients available for {task}.')
    if len(features) == 0:
        raise ValueError(f'No features available for {task}.')
    raw = shap_benefit[task].reindex(index=patients, columns=features).T
    scaled, denom = _row_scale_full_heatmap(raw, quantile=0.95)
    if 'MBP_raw' in patient_order_df.columns and len(patient_order_df) > 0:
        mbp_series = patient_order_df.reindex(patients)['MBP_raw']
    else:
        mbp_series = pd.Series(index=patients, dtype=object)
    modal_map = feature_preference[['feature_clean', 'modal_clean']].drop_duplicates('feature_clean').set_index('feature_clean')['modal_clean'].to_dict()
    fig = plt.figure(figsize=(12.6, max(10.6, 0.255 * len(features) + 2.3)), facecolor='white')
    gs = fig.add_gridspec(nrows=2, ncols=2, height_ratios=[0.12, 1.0], width_ratios=[0.05, 1.0], hspace=0.035, wspace=0.025)
    ax_blank = fig.add_subplot(gs[0, 0])
    ax_top = fig.add_subplot(gs[0, 1])
    ax_strip = fig.add_subplot(gs[1, 0])
    ax = fig.add_subplot(gs[1, 1])
    ax_blank.axis('off')
    rgba = np.ones((1, len(patients), 4), dtype=float)
    for j, p in enumerate(patients):
        grp = mbp_series.get(p, np.nan)
        rgba[0, j, :] = mpl.colors.to_rgba(MBP_COLORS.get(grp, '#E7E7E7'))
    ax_top.imshow(rgba, aspect='auto', interpolation='nearest')
    ax_top.set_xticks([])
    ax_top.set_yticks([])
    for sp in ax_top.spines.values():
        sp.set_visible(False)
    feature_rgba = np.array([mpl.colors.to_rgba(MODALITY_COLORS.get(modal_map.get(f, ''), '#AAAAAA')) for f in features]).reshape(len(features), 1, 4)
    ax_strip.imshow(feature_rgba, aspect='auto', interpolation='nearest')
    ax_strip.set_xticks([])
    ax_strip.set_yticks([])
    for sp in ax_strip.spines.values():
        sp.set_visible(False)
    im = ax.imshow(scaled.values, aspect='auto', interpolation='nearest', cmap=BENEFIT_CMAP, vmin=-1, vmax=1)
    ax.set_xticks([])
    ax.set_yticks(np.arange(len(features)))
    ax.set_yticklabels([clean_feature_label(f, 44) for f in features], fontsize=9.2)
    ax.tick_params(axis='y', length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_title(task, fontsize=11.8, fontweight='bold', color=ENDPOINT_COLORS[task], pad=7)
    if mbp_series.notna().any():
        groups = mbp_series.values
        for j in range(1, len(groups)):
            if groups[j] != groups[j - 1]:
                ax.axvline(j - 0.5, color='white', linewidth=1.15, zorder=5)
                ax_top.axvline(j - 0.5, color='white', linewidth=1.15, zorder=5)
    cbar = fig.colorbar(im, ax=ax, fraction=0.018, pad=0.015)
    cbar.set_label('Row-scaled local SHAP\n(benefit-oriented)', fontsize=9.7)
    cbar.ax.tick_params(labelsize=6.3)
    cbar.outline.set_visible(False)
    mbp_present = [g for g in MBP_ORDER if g in set(mbp_series.dropna())]
    mbp_handles = [Line2D([0], [0], marker='s', linestyle='', markerfacecolor=MBP_COLORS[g], markeredgecolor='white', markersize=7.0, label=g) for g in mbp_present]
    modality_handles = [Line2D([0], [0], marker='s', linestyle='', markerfacecolor=MODALITY_COLORS[m], markeredgecolor='white', markersize=7.0, label=m) for m in MODALITY_ORDER_MRU]
    if len(mbp_handles) > 0:
        leg1 = fig.legend(handles=mbp_handles, title='Patient group', loc='upper center', bbox_to_anchor=(0.58, 0.945), ncol=len(mbp_handles), frameon=False, fontsize=9.7, title_fontsize=9.7)
    fig.legend(handles=modality_handles, title='Feature modality', loc='upper right', bbox_to_anchor=(0.965, 0.945), ncol=3, frameon=False, fontsize=9.7, title_fontsize=9.7)
    add_panel_header(fig, panel, f'Complete patient-level MRU attribution landscape for {task}')
    raw.to_csv(FULL_HEATMAP_SOURCE_DIR / f"{panel}_{task}{get_value('science.tables.attribution_034', 'attribution_034.csv')}", encoding='utf-8-sig')
    scaled.to_csv(FULL_HEATMAP_SOURCE_DIR / f"{panel}_{task}{get_value('science.tables.attribution_035', 'attribution_035.csv')}", encoding='utf-8-sig')
    pd.DataFrame({'sample': patients, 'MBP_raw': mbp_series.reindex(patients).values}).to_csv(FULL_HEATMAP_SOURCE_DIR / f"{panel}_{task}{get_value('science.tables.result_220', 'result_220.csv')}", index=False, encoding='utf-8-sig')
    pd.DataFrame({'feature': features, 'modality': [modal_map.get(f, '') for f in features], 'row_scale_denominator': [denom.get(f, np.nan) for f in features]}).to_csv(FULL_HEATMAP_SOURCE_DIR / f"{panel}_{task}{get_value('science.tables.result_221', 'result_221.csv')}", index=False, encoding='utf-8-sig')
    fig.subplots_adjust(left=0.34, right=0.94, top=0.88, bottom=0.05)
    save_figure(fig, FULL_HEATMAP_OUT_DIR / f'{panel}_{task}_full_patient_SHAP_heatmap_Nature_style')
    plt.show()
    return scaled
sxl_response = plot_full_patient_shap_heatmap('Response')
sxm_pfs = plot_full_patient_shap_heatmap('PFS')
sxn_os = plot_full_patient_shap_heatmap('OS')





In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
DEPENDENCE_OUT_DIR = SUPP_OUT_DIR / get_value('science.paths.directory_158', 'directory_158')
DEPENDENCE_SOURCE_DIR = SOURCE_OUT_DIR / get_value('science.paths.directory_158', 'directory_158')
DEPENDENCE_ATLAS_DIR = DEPENDENCE_OUT_DIR / get_value('science.paths.directory_159', 'directory_159')
for d in [DEPENDENCE_OUT_DIR, DEPENDENCE_SOURCE_DIR, DEPENDENCE_ATLAS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
DEPENDENCE_PANEL = {'Metagenomics': 'SxO', 'Radiomics': 'SxP', 'Urine': 'SxQ'}
MAKE_FULL_DEPENDENCE_ATLAS = True
DEPENDENCE_BINS = 8

def _safe_spearman_with_p(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 5 or len(np.unique(x)) < 3 or len(np.unique(y)) < 3:
        return (np.nan, np.nan)
    try:
        from scipy.stats import spearmanr
        rho, p = spearmanr(x, y)
        return (float(rho), float(p))
    except Exception:
        rho = pd.Series(x).corr(pd.Series(y), method='spearman')
        return (float(rho), np.nan)

def _format_p(p):
    if not np.isfinite(p):
        return 'P=NA'
    if p < 0.0001:
        return 'P<1×10⁻⁴'
    if p < 0.001:
        return f'P={p:.1e}'
    return f'P={p:.3f}'

def _quantile_median_trend(x, y, n_bins=DEPENDENCE_BINS):
    tmp = pd.DataFrame({'x': pd.to_numeric(pd.Series(x), errors='coerce'), 'y': pd.to_numeric(pd.Series(y), errors='coerce')}).dropna()
    if len(tmp) < 10 or tmp['x'].nunique() < 4:
        return pd.DataFrame(columns=['x_median', 'y_median', 'n'])
    try:
        q = min(n_bins, int(tmp['x'].nunique()))
        tmp['bin'] = pd.qcut(tmp['x'], q=q, duplicates='drop')
    except Exception:
        return pd.DataFrame(columns=['x_median', 'y_median', 'n'])
    trend = tmp.groupby('bin', observed=True).agg(x_median=('x', 'median'), y_median=('y', 'median'), n=('y', 'size')).reset_index(drop=True).sort_values('x_median')
    return trend

def _dependence_feature_role_table():
    rows = []
    for modal in MODALITY_ORDER_MRU:
        ss = feature_preference[feature_preference['modal_clean'] == modal].copy()
        if ss.empty:
            continue
        selected = []
        shared = ss.sort_values(['shared_score', 'overall_relative_importance'], ascending=False).iloc[0]
        selected.append(str(shared['feature_clean']))
        rows.append({'modal_clean': modal, 'role': 'Shared backbone', 'feature_clean': str(shared['feature_clean']), 'selection_score': float(shared['shared_score'])})
        for task in ENDPOINT_ORDER:
            score_col = f'{task}_preference_score'
            cand = ss[~ss['feature_clean'].astype(str).isin(selected)].sort_values([score_col, 'overall_relative_importance'], ascending=False)
            positive = cand[pd.to_numeric(cand[score_col], errors='coerce').fillna(0) > 0]
            if not positive.empty:
                chosen = positive.iloc[0]
            else:
                cand2 = cand[cand['dominant_endpoint'].astype(str) == task].copy()
                if not cand2.empty:
                    cand2['_fallback_score'] = cand2['preference_margin'].fillna(0) * cand2['overall_relative_importance'].fillna(0)
                    chosen = cand2.sort_values(['_fallback_score', 'overall_relative_importance'], ascending=False).iloc[0]
                else:
                    chosen = cand.iloc[0]
            feat = str(chosen['feature_clean'])
            selected.append(feat)
            rows.append({'modal_clean': modal, 'role': f'{task}-preferred', 'feature_clean': feat, 'selection_score': float(chosen.get(score_col, np.nan))})
    table = pd.DataFrame(rows)
    table.to_csv(DEPENDENCE_SOURCE_DIR / get_value('science.tables.attribution_036', 'attribution_036.csv'), index=False, encoding='utf-8-sig')
    return table
dependence_feature_roles = _dependence_feature_role_table()
display(dependence_feature_roles)

def _plot_one_dependence_panel(ax, feature, task, endpoint_color, show_xlabel=True, show_ylabel=True):
    common = X_mru.index.intersection(shap_benefit[task].index)
    x = pd.to_numeric(X_mru.loc[common, feature], errors='coerce')
    y = pd.to_numeric(shap_benefit[task].loc[common, feature], errors='coerce')
    mask = x.notna() & y.notna()
    x = x.loc[mask].astype(float)
    y = y.loc[mask].astype(float)
    rho, p = _safe_spearman_with_p(x.values, y.values)
    ax.scatter(x.values, y.values, s=37, facecolor=endpoint_color, edgecolor='white', linewidth=0.35, alpha=0.98, zorder=3)
    ax.axhline(0, color='#9A9A9A', linewidth=0.75, linestyle=(0, (3, 3)), zorder=1)
    trend = _quantile_median_trend(x.values, y.values, n_bins=DEPENDENCE_BINS)
    if len(trend) >= 2:
        ax.plot(trend['x_median'], trend['y_median'], color='#222222', linewidth=1.35, marker='o', markersize=5.5, markerfacecolor='white', markeredgecolor='#222222', markeredgewidth=0.65, zorder=4)
    ax.text(0.03, 0.97, f'ρ={rho:.2f}', transform=ax.transAxes, ha='left', va='top', fontsize=9.2, color=COLOR_TEXT, bbox=dict(boxstyle='round,pad=0.20', facecolor='white', edgecolor='none', alpha=0.98), zorder=5)
    if show_xlabel:
        ax.set_xlabel('Feature value', fontsize=9.7)
    else:
        ax.set_xlabel('')
    if show_ylabel:
        ax.set_ylabel('Benefit-oriented SHAP', fontsize=9.7)
    else:
        ax.set_ylabel('')
    ax.tick_params(labelsize=6.4)
    clean_axes(ax, grid_axis='both')
    return {'rho': rho, 'p': p, 'n': int(len(x)), 'trend': trend, 'sample_ids': x.index.tolist(), 'x': x.values, 'y': y.values}

def plot_modality_dependence_figure(modal):
    if modal not in DEPENDENCE_PANEL:
        raise ValueError(modal)
    panel = DEPENDENCE_PANEL[modal]
    ss = dependence_feature_roles[dependence_feature_roles['modal_clean'] == modal].copy()
    role_order = ['Shared backbone', 'Response-preferred', 'PFS-preferred', 'OS-preferred']
    ss['role'] = pd.Categorical(ss['role'], categories=role_order, ordered=True)
    ss = ss.sort_values('role')
    features = ss['feature_clean'].tolist()
    roles = ss['role'].astype(str).tolist()
    n_rows = len(features)
    fig, axes = plt.subplots(n_rows, 3, figsize=(11.2, max(8.2, 2.25 * n_rows)), facecolor='white', squeeze=False)
    source_rows = []
    trend_rows = []
    for r, (feature, role) in enumerate(zip(features, roles)):
        for c, task in enumerate(ENDPOINT_ORDER):
            ax = axes[r, c]
            stat = _plot_one_dependence_panel(ax=ax, feature=feature, task=task, endpoint_color=ENDPOINT_COLORS[task], show_xlabel=r == n_rows - 1, show_ylabel=c == 0)
            if r == 0:
                ax.set_title(task, fontsize=11.8, fontweight='bold', color=ENDPOINT_COLORS[task], pad=7)
            if c == 0:
                ax.text(-0.34, 0.5, f'{role}\n{clean_feature_label(feature, 34)}', transform=ax.transAxes, ha='right', va='center', fontsize=9.7, color=COLOR_TEXT, fontweight='bold' if role == 'Shared backbone' else 'normal', clip_on=False)
            for sid, xv, yv in zip(stat['sample_ids'], stat['x'], stat['y']):
                source_rows.append({'panel': panel, 'modal_clean': modal, 'role': role, 'feature_clean': feature, 'outcome_clean': task, 'sample': sid, 'feature_value': float(xv), 'benefit_oriented_shap': float(yv), 'spearman_rho': stat['rho'], 'spearman_p': stat['p']})
            if len(stat['trend']) > 0:
                tt = stat['trend'].copy()
                tt['panel'] = panel
                tt['modal_clean'] = modal
                tt['role'] = role
                tt['feature_clean'] = feature
                tt['outcome_clean'] = task
                trend_rows.append(tt)
    modality_display = modal
    add_panel_header(fig, panel, f'Representative MRU dependence profiles — {modality_display}')
    source_df = pd.DataFrame(source_rows)
    source_df.to_csv(DEPENDENCE_SOURCE_DIR / f"{panel}_{modal}{get_value('science.tables.attribution_037', 'attribution_037.csv')}", index=False, encoding='utf-8-sig')
    if trend_rows:
        pd.concat(trend_rows, axis=0, ignore_index=True).to_csv(DEPENDENCE_SOURCE_DIR / f"{panel}_{modal}{get_value('science.tables.attribution_038', 'attribution_038.csv')}", index=False, encoding='utf-8-sig')
    fig.subplots_adjust(left=0.28, right=0.98, top=0.9, bottom=0.06, wspace=0.25, hspace=0.35)
    save_figure(fig, DEPENDENCE_OUT_DIR / f'{panel}_{modal}_representative_dependence_Nature_style')
    plt.show()
    return source_df
sxo_metagenomics = plot_modality_dependence_figure('Metagenomics')
sxp_radiomics = plot_modality_dependence_figure('Radiomics')
sxq_urine = plot_modality_dependence_figure('Urine')

def _plot_atlas_single_panel(ax, feature, task, rank):
    stat = _plot_one_dependence_panel(ax=ax, feature=feature, task=task, endpoint_color=ENDPOINT_COLORS[task], show_xlabel=True, show_ylabel=True)
    modal = feature_preference.drop_duplicates('feature_clean').set_index('feature_clean').reindex([feature])['modal_clean'].iloc[0] if feature in set(feature_preference['feature_clean']) else ''
    modal_display = modal
    ax.set_title(f'#{rank} | {modal_display}\n{clean_feature_label(feature, 34)}', fontsize=10.0, color=COLOR_TEXT, pad=5)
    return stat

def make_full_dependence_atlas(task, plots_per_page=12):
    task_imp = determinant_long[determinant_long['outcome_clean'] == task].sort_values('mean_abs_shap_clean', ascending=False)
    features = [f for f in task_imp['feature_clean'].tolist() if f in X_mru.columns and f in shap_benefit[task].columns]
    features = list(dict.fromkeys(features))
    n_cols = 3
    n_rows = int(np.ceil(plots_per_page / n_cols))
    n_pages = int(np.ceil(len(features) / plots_per_page))
    pdf_path = DEPENDENCE_ATLAS_DIR / f'Extended_all39_dependence_{task}.pdf'
    summary_rows = []
    with PdfPages(pdf_path) as pdf:
        for page in range(n_pages):
            start = page * plots_per_page
            end = min((page + 1) * plots_per_page, len(features))
            page_features = features[start:end]
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(11.5, 3.15 * n_rows), facecolor='white', squeeze=False)
            for i, feature in enumerate(page_features):
                r = i // n_cols
                c = i % n_cols
                rank = start + i + 1
                stat = _plot_atlas_single_panel(axes[r, c], feature, task, rank)
                summary_rows.append({'outcome_clean': task, 'rank_by_mean_abs_shap': rank, 'feature_clean': feature, 'spearman_rho': stat['rho'], 'spearman_p': stat['p'], 'n': stat['n']})
            for j in range(len(page_features), n_rows * n_cols):
                r = j // n_cols
                c = j % n_cols
                axes[r, c].axis('off')
            fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05, wspace=0.3, hspace=0.48)
            pdf.savefig(fig, bbox_inches='tight', facecolor='white')
            plt.close(fig)
    return (pd.DataFrame(summary_rows), pdf_path)
if MAKE_FULL_DEPENDENCE_ATLAS:
    atlas_summaries = []
    for task in ENDPOINT_ORDER:
        summary_df, pdf_path = make_full_dependence_atlas(task)
        atlas_summaries.append(summary_df)
        
    atlas_summary = pd.concat(atlas_summaries, axis=0, ignore_index=True)
    atlas_summary.to_csv(DEPENDENCE_SOURCE_DIR / get_value('science.tables.attribution_039', 'attribution_039.csv'), index=False, encoding='utf-8-sig')





In [ ]:
SXR_OUT_DIR = SUPP_OUT_DIR / get_value('science.paths.directory_160', 'directory_160')
SXR_SOURCE_DIR = SOURCE_OUT_DIR / get_value('science.paths.directory_160', 'directory_160')
for d in [SXR_OUT_DIR, SXR_SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
PAIR_ORDER = ['Response–PFS', 'Response–OS', 'PFS–OS']
PAIR_TUPLES = [('Response', 'PFS'), ('Response', 'OS'), ('PFS', 'OS')]

def _build_modality_concordance():
    feature_modal = feature_preference[['feature_clean', 'modal_clean']].drop_duplicates('feature_clean')
    rows = []
    common_patients = set(shap_benefit['Response'].index)
    for task in ['PFS', 'OS']:
        common_patients &= set(shap_benefit[task].index)
    ordered_patients = [p for p in PATIENT_ORDER if p in common_patients]
    for modal in MODALITY_ORDER_MRU:
        features = feature_modal.loc[feature_modal['modal_clean'] == modal, 'feature_clean'].astype(str).tolist()
        features = [f for f in features if all((f in shap_benefit[t].columns for t in ENDPOINT_ORDER))]
        if len(features) < 3:
            continue
        for patient in ordered_patients:
            grp = patient_order_df.loc[patient, 'MBP_raw'] if patient in patient_order_df.index and 'MBP_raw' in patient_order_df.columns else np.nan
            for a, b in PAIR_TUPLES:
                rho = _spearman_pair(shap_benefit[a].loc[patient, features].values, shap_benefit[b].loc[patient, features].values)
                rows.append({'sample': patient, 'pair': f'{a}–{b}', 'endpoint_a': a, 'endpoint_b': b, 'modal_clean': modal, 'spearman_rho': rho, 'MBP_raw': grp, 'n_features': len(features)})
    d = pd.DataFrame(rows)
    d.to_csv(SXR_SOURCE_DIR / get_value('science.tables.result_222', 'result_222.csv'), index=False, encoding='utf-8-sig')
    return d
sxr_modality_concordance = _build_modality_concordance()

def _summary_rho_table(d, strata_col, strata_order):
    rows = []
    for stratum in strata_order:
        for pair in PAIR_ORDER:
            vals = d.loc[(d[strata_col] == stratum) & (d['pair'] == pair), 'spearman_rho'].dropna().astype(float).values
            if len(vals) == 0:
                continue
            rows.append({'strata_type': strata_col, 'stratum': stratum, 'pair': pair, 'n': int(len(vals)), 'median_rho': float(np.nanmedian(vals)), 'q25': float(np.nanquantile(vals, 0.25)), 'q75': float(np.nanquantile(vals, 0.75)), 'mean_rho': float(np.nanmean(vals)), 'sd_rho': float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else np.nan})
    return pd.DataFrame(rows)
sxr_mbp_summary = _summary_rho_table(patient_concordance, 'MBP_raw', MBP_ORDER)
sxr_modality_summary = _summary_rho_table(sxr_modality_concordance, 'modal_clean', MODALITY_ORDER_MRU)
print('\nSxR modality summary columns:', sxr_modality_summary.columns.tolist())
print(sxr_modality_summary.head())
sxr_overall_summary = patient_concordance.groupby('pair', as_index=False).agg(n=('spearman_rho', lambda s: int(s.notna().sum())), median_rho=('spearman_rho', 'median'), q25=('spearman_rho', lambda s: s.quantile(0.25)), q75=('spearman_rho', lambda s: s.quantile(0.75)), mean_rho=('spearman_rho', 'mean'), sd_rho=('spearman_rho', 'std'))
sxr_overall_summary['strata_type'] = 'Overall'
sxr_overall_summary['stratum'] = 'Overall'
sxr_stats = pd.concat([sxr_overall_summary[['strata_type', 'stratum', 'pair', 'n', 'median_rho', 'q25', 'q75', 'mean_rho', 'sd_rho']], sxr_mbp_summary, sxr_modality_summary], axis=0, ignore_index=True)
sxr_stats.to_csv(SXR_SOURCE_DIR / get_value('science.tables.metrics_077', 'metrics_077.csv'), index=False, encoding='utf-8-sig')

def plot_sxr_extended_concordance():
    fig = plt.figure(figsize=(12.2, 7.8), facecolor='white')
    gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 0.88], width_ratios=[1.15, 0.85], hspace=0.4, wspace=0.3)
    ax_a = fig.add_subplot(gs[0, 0])
    x_base = np.arange(len(PAIR_ORDER))
    offsets = {'MBP-I': -0.24, 'MBP-II': 0.0, 'MBP-III': 0.24}
    rng = np.random.default_rng(20260810)
    for grp in MBP_ORDER:
        if grp not in set(patient_concordance['MBP_raw'].dropna()):
            continue
        for i, pair in enumerate(PAIR_ORDER):
            vals = patient_concordance.loc[(patient_concordance['MBP_raw'] == grp) & (patient_concordance['pair'] == pair), 'spearman_rho'].dropna().astype(float).values
            if len(vals) == 0:
                continue
            xpos = i + offsets[grp]
            vp = ax_a.violinplot(vals, positions=[xpos], widths=0.2, showextrema=False, showmeans=False, showmedians=False)
            for body in vp['bodies']:
                body.set_facecolor(MBP_COLORS[grp])
                body.set_edgecolor(MBP_COLORS[grp])
                body.set_alpha(0.22)
                body.set_linewidth(0.6)
            jitter = rng.uniform(-0.035, 0.035, size=len(vals))
            ax_a.scatter(xpos + jitter, vals, s=23, facecolor=MBP_COLORS[grp], edgecolor='white', linewidth=0.7, alpha=0.98, zorder=3)
            med = float(np.nanmedian(vals))
            q25 = float(np.nanquantile(vals, 0.25))
            q75 = float(np.nanquantile(vals, 0.75))
            ax_a.plot([xpos - 0.055, xpos + 0.055], [med, med], color=COLOR_TEXT, lw=1.5, zorder=5)
            ax_a.plot([xpos, xpos], [q25, q75], color=COLOR_TEXT, lw=0.9, zorder=5)
    ax_a.set_xticks(x_base)
    ax_a.set_xticklabels(PAIR_ORDER)
    ax_a.set_ylabel('Within-patient Spearman ρ\nacross MRU feature SHAP profiles')
    ax_a.set_ylim(0.55, 1.015)
    ax_a.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])
    clean_axes(ax_a, grid_axis='y')
    handles = [Line2D([0], [0], marker='o', linestyle='', markerfacecolor=MBP_COLORS[g], markeredgecolor='white', markersize=7.0, label=g) for g in MBP_ORDER if g in set(patient_concordance['MBP_raw'].dropna())]
    if handles:
        ax_a.legend(handles=handles, title='MBP group', frameon=False, loc='lower left', fontsize=9.7, title_fontsize=9.7, ncol=len(handles))
    ax_a.set_title('Concordance by MBP group', loc='left', fontsize=11.8, fontweight='bold', pad=8)
    ax_b = fig.add_subplot(gs[0, 1])
    med_mat = sxr_modality_summary.pivot_table(index='stratum', columns='pair', values='median_rho', aggfunc='first').reindex(index=MODALITY_ORDER_MRU, columns=PAIR_ORDER)
    im = ax_b.imshow(med_mat.values, aspect='auto', interpolation='nearest', cmap='Blues', vmin=0.55, vmax=1.0)
    ax_b.set_xticks(np.arange(len(PAIR_ORDER)))
    ax_b.set_xticklabels(PAIR_ORDER, rotation=35, ha='right', fontsize=9.7)
    ax_b.set_yticks(np.arange(len(MODALITY_ORDER_MRU)))
    ax_b.set_yticklabels(MODALITY_ORDER_MRU, fontsize=9.7)
    ax_b.tick_params(length=0)
    for i, modal in enumerate(MODALITY_ORDER_MRU):
        for j, pair in enumerate(PAIR_ORDER):
            val = med_mat.loc[modal, pair]
            if np.isfinite(val):
                row = sxr_modality_summary[(sxr_modality_summary['stratum'] == modal) & (sxr_modality_summary['pair'] == pair)]
                if len(row) > 0:
                    q25 = float(row['q25'].iloc[0])
                    q75 = float(row['q75'].iloc[0])
                    txt = f'{val:.2f}\n[{q25:.2f}, {q75:.2f}]'
                else:
                    txt = f'{val:.2f}'
                ax_b.text(j, i, txt, ha='center', va='center', fontsize=9.2, color='white' if val >= 0.8 else COLOR_TEXT, fontweight='bold' if val >= 0.9 else 'normal')
    for i, modal in enumerate(MODALITY_ORDER_MRU):
        ax_b.scatter(-0.65, i, s=49, marker='s', facecolor=MODALITY_COLORS.get(modal, '#999999'), edgecolor='white', linewidth=0.75, clip_on=False)
    for sp in ax_b.spines.values():
        sp.set_visible(False)
    cbar = fig.colorbar(im, ax=ax_b, fraction=0.045, pad=0.04)
    cbar.set_label('Median within-patient ρ', fontsize=9.7)
    cbar.ax.tick_params(labelsize=6.3)
    cbar.outline.set_visible(False)
    ax_b.set_title('Concordance within each feature modality', loc='left', fontsize=11.8, fontweight='bold', pad=8)
    ax_c = fig.add_subplot(gs[1, :])
    strata_rows = [('Overall', 'Overall'), ('MBP-I', 'MBP_raw'), ('MBP-II', 'MBP_raw'), ('MBP-III', 'MBP_raw'), ('Radiomics', 'modal_clean'), ('Metagenomics', 'modal_clean'), ('Urine', 'modal_clean')]
    summary_matrix = np.full((len(strata_rows), len(PAIR_ORDER)), np.nan)
    summary_text = [[''] * len(PAIR_ORDER) for _ in strata_rows]
    for i, (stratum, strata_type) in enumerate(strata_rows):
        for j, pair in enumerate(PAIR_ORDER):
            if stratum == 'Overall':
                rr = sxr_stats[(sxr_stats['stratum'] == 'Overall') & (sxr_stats['pair'] == pair)]
            else:
                rr = sxr_stats[(sxr_stats['stratum'] == stratum) & (sxr_stats['pair'] == pair)]
            if len(rr) == 0:
                continue
            med = float(rr['median_rho'].iloc[0])
            q25 = float(rr['q25'].iloc[0])
            q75 = float(rr['q75'].iloc[0])
            n = int(rr['n'].iloc[0])
            summary_matrix[i, j] = med
            summary_text[i][j] = f'{med:.2f} [{q25:.2f}, {q75:.2f}]\nn={n}'
    im2 = ax_c.imshow(summary_matrix, aspect='auto', interpolation='nearest', cmap='PuBuGn', vmin=0.55, vmax=1.0)
    ax_c.set_xticks(np.arange(len(PAIR_ORDER)))
    ax_c.set_xticklabels(PAIR_ORDER, fontsize=9.7)
    ax_c.set_yticks(np.arange(len(strata_rows)))
    ax_c.set_yticklabels([x[0] for x in strata_rows], fontsize=9.7)
    ax_c.tick_params(length=0)
    for i in range(len(strata_rows)):
        for j in range(len(PAIR_ORDER)):
            val = summary_matrix[i, j]
            if np.isfinite(val):
                ax_c.text(j, i, summary_text[i][j], ha='center', va='center', fontsize=9.2, color='white' if val >= 0.82 else COLOR_TEXT, fontweight='bold' if i == 0 else 'normal')
    for sp in ax_c.spines.values():
        sp.set_visible(False)
    ax_c.set_title('Complete descriptive summary: median ρ [IQR]', loc='left', fontsize=11.8, fontweight='bold', pad=8)
    cbar2 = fig.colorbar(im2, ax=ax_c, fraction=0.018, pad=0.012)
    cbar2.set_label('Median within-patient ρ', fontsize=9.7)
    cbar2.ax.tick_params(labelsize=6.3)
    cbar2.outline.set_visible(False)
    add_panel_header(fig, 'SxR', 'Extended cross-endpoint concordance of patient-level MRU attribution profiles')
    fig.subplots_adjust(left=0.11, right=0.95, top=0.89, bottom=0.06)
    save_figure(fig, SXR_OUT_DIR / get_value('science.paths.directory_161', 'directory_161'))
    plt.show()
    return sxr_stats
sxr_output = plot_sxr_extended_concordance()


In [ ]:
LOCAL_OUT_DIR = SUPP_OUT_DIR / get_value('science.paths.directory_162', 'directory_162')
LOCAL_SOURCE_DIR = SOURCE_OUT_DIR / get_value('science.paths.directory_162', 'directory_162')
for d in [LOCAL_OUT_DIR, LOCAL_SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
LOCAL_PANEL = {'MBP-I': 'SxS', 'MBP-II': 'SxT', 'MBP-III': 'SxU'}
TOP_LOCAL_FEATURES = 9

def _build_robust_local_profile_matrix():
    common_patients = set(PATIENT_ORDER)
    for task in ENDPOINT_ORDER:
        common_patients &= set(shap_benefit[task].index)
    patients = [p for p in PATIENT_ORDER if p in common_patients]
    blocks = []
    for task in ENDPOINT_ORDER:
        features = [f for f in X_mru.columns if f in shap_benefit[task].columns]
        B = shap_benefit[task].reindex(index=patients, columns=features).copy()
        B.columns = [f'{task}::{c}' for c in B.columns]
        blocks.append(B)
    Z = pd.concat(blocks, axis=1).astype(float)
    Zs = Z.copy()
    for c in Z.columns:
        x = pd.to_numeric(Z[c], errors='coerce').astype(float)
        med = float(np.nanmedian(x))
        mad = float(np.nanmedian(np.abs(x - med)))
        scale = 1.4826 * mad
        if not np.isfinite(scale) or scale <= 1e-12:
            q25, q75 = np.nanquantile(x, [0.25, 0.75])
            scale = float((q75 - q25) / 1.349)
        if not np.isfinite(scale) or scale <= 1e-12:
            scale = float(np.nanstd(x))
        if not np.isfinite(scale) or scale <= 1e-12:
            scale = 1.0
        Zs[c] = (x - med) / scale
    return (Z, Zs)
local_profile_raw, local_profile_scaled = _build_robust_local_profile_matrix()

def select_representative_patients():
    if 'MBP_raw' not in patient_order_df.columns or patient_order_df['MBP_raw'].notna().sum() == 0:
        raise ValueError('MBP annotations are required for SxS–U.')
    rows = []
    for grp in MBP_ORDER:
        ids = patient_order_df.index[patient_order_df['MBP_raw'] == grp].intersection(local_profile_scaled.index).tolist()
        if len(ids) == 0:
            continue
        G = local_profile_scaled.loc[ids].replace([np.inf, -np.inf], np.nan)
        centroid = G.median(axis=0)
        distances = {}
        for p in ids:
            v = G.loc[p]
            mask = v.notna() & centroid.notna()
            if mask.sum() == 0:
                dist = np.inf
            else:
                diff = (v.loc[mask] - centroid.loc[mask]).values.astype(float)
                dist = float(np.sqrt(np.mean(diff ** 2)))
            distances[p] = dist
        representative = min(distances, key=distances.get)
        row = {'MBP_raw': grp, 'sample': representative, 'robust_distance_to_group_median': distances[representative], 'n_group_patients': len(ids)}
        if representative in patient_order_df.index and 'composite_benefit_rank' in patient_order_df.columns:
            row['composite_benefit_rank'] = float(patient_order_df.loc[representative, 'composite_benefit_rank'])
        rows.append(row)
    out = pd.DataFrame(rows)
    out.to_csv(LOCAL_SOURCE_DIR / get_value('science.tables.result_223', 'result_223.csv'), index=False, encoding='utf-8-sig')
    return out
representative_patients = select_representative_patients()
display(representative_patients)

def _get_local_contribution_table(patient, task, top_n=TOP_LOCAL_FEATURES):
    features = [f for f in X_mru.columns if f in shap_benefit[task].columns]
    s = pd.to_numeric(shap_benefit[task].loc[patient, features], errors='coerce').astype(float)
    s = s.dropna()
    modal_map = feature_preference[['feature_clean', 'modal_clean']].drop_duplicates('feature_clean').set_index('feature_clean')['modal_clean'].to_dict()
    ranked = s.abs().sort_values(ascending=False).index.tolist()
    top_features = ranked[:top_n]
    other_features = [f for f in s.index if f not in top_features]
    rows = []
    for feat in top_features:
        xval = float(pd.to_numeric(pd.Series([X_mru.loc[patient, feat]]), errors='coerce').iloc[0]) if feat in X_mru.columns else np.nan
        rows.append({'feature': feat, 'display_feature': clean_feature_label(feat, 34), 'modal_clean': modal_map.get(feat, 'Unknown'), 'feature_value': xval, 'benefit_oriented_shap': float(s.loc[feat]), 'abs_shap': float(abs(s.loc[feat])), 'component_type': 'feature'})
    if len(other_features) > 0:
        other_sum = float(s.loc[other_features].sum())
        rows.append({'feature': 'Other MRU determinants', 'display_feature': f'Other {len(other_features)} determinants', 'modal_clean': 'Other', 'feature_value': np.nan, 'benefit_oriented_shap': other_sum, 'abs_shap': abs(other_sum), 'component_type': 'other'})
    d = pd.DataFrame(rows)
    feat_part = d[d['component_type'] == 'feature'].sort_values('abs_shap', ascending=False)
    other_part = d[d['component_type'] == 'other']
    d = pd.concat([feat_part, other_part], axis=0, ignore_index=True)
    cumulative = [0.0]
    starts = []
    ends = []
    cur = 0.0
    for v in d['benefit_oriented_shap'].values.astype(float):
        starts.append(cur)
        cur = cur + v
        ends.append(cur)
        cumulative.append(cur)
    d['waterfall_start'] = starts
    d['waterfall_end'] = ends
    d['cumulative_after'] = ends
    d['total_local_shap_sum'] = cur
    return d

def _projected_value_text(patient, task):
    if patient not in pred_mru.index:
        return ''
    if task == 'Response':
        col = find_col(pred_mru, ['Response_pred', 'response_pred', 'Response_probability'], required=False)
        if col is not None:
            val = float(pd.to_numeric(pd.Series([pred_mru.loc[patient, col]]), errors='coerce').iloc[0])
            return f'projected response={val:.2f}' if np.isfinite(val) else ''
    elif task == 'PFS':
        col = find_col(pred_mru, ['PFS_risk', 'pfs_risk'], required=False)
        if col is not None:
            val = float(pd.to_numeric(pd.Series([pred_mru.loc[patient, col]]), errors='coerce').iloc[0])
            return f'model risk={val:.2f}' if np.isfinite(val) else ''
    elif task == 'OS':
        col = find_col(pred_mru, ['OS_risk', 'os_risk'], required=False)
        if col is not None:
            val = float(pd.to_numeric(pd.Series([pred_mru.loc[patient, col]]), errors='coerce').iloc[0])
            return f'model risk={val:.2f}' if np.isfinite(val) else ''
    return ''

def _plot_local_waterfall_axis(ax, d, patient, task):
    dd = d.copy().reset_index(drop=True)
    y = np.arange(len(dd))
    positive_color = '#D95F59'
    negative_color = '#3B82B8'
    other_color = '#A7A7A7'
    for i, r in dd.iterrows():
        start = float(r['waterfall_start'])
        end = float(r['waterfall_end'])
        val = float(r['benefit_oriented_shap'])
        left = min(start, end)
        width = abs(end - start)
        if r['component_type'] == 'other':
            color = other_color
        else:
            color = positive_color if val >= 0 else negative_color
        ax.barh(i, width, left=left, height=0.68, color=color, edgecolor='white', linewidth=0.55, alpha=0.98, zorder=3)
        if i < len(dd) - 1:
            ax.plot([end, end], [i + 0.34, i + 1 - 0.34], color='#B7B7B7', linewidth=0.95, linestyle=(0, (2, 2)), zorder=2)
        txt = f'{val:+.3g}'
        ax.text(end, i, txt, ha='left' if val >= 0 else 'right', va='center', fontsize=8.8, color=COLOR_TEXT, clip_on=False)
    labels = []
    for _, r in dd.iterrows():
        if r['component_type'] == 'other':
            labels.append(str(r['display_feature']))
        else:
            modality = str(r['modal_clean'])
            labels.append(clean_feature_label(r['display_feature'], 31))
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9.2)
    ax.invert_yaxis()
    ax.axvline(0, color='#777777', linewidth=0.75, linestyle=(0, (3, 3)), zorder=1)
    xlim = ax.get_xlim()
    x_span = xlim[1] - xlim[0]
    x_square = xlim[0] - 0.055 * x_span
    for i, r in dd.iterrows():
        modal = str(r['modal_clean'])
        if modal == 'Other':
            c = other_color
        else:
            c = MODALITY_COLORS.get(modal, '#AAAAAA')
        ax.scatter(x_square, i, s=40, marker='s', facecolor=c, edgecolor='white', linewidth=0.75, clip_on=False, zorder=5)
    ax.set_xlim(xlim)
    total = float(dd['total_local_shap_sum'].iloc[0])
    ax.scatter(total, -0.72, marker='D', s=43, facecolor=ENDPOINT_COLORS[task], edgecolor='white', linewidth=0.9, clip_on=False, zorder=6)
    ax.text(total, -0.72, f'  ΣSHAP={total:+.3g}', ha='left', va='center', fontsize=8.8, color=ENDPOINT_COLORS[task], fontweight='bold', clip_on=False)
    proj_text = _projected_value_text(patient, task)
    title = task
    if proj_text:
        title = f'{task}\n{proj_text}'
    ax.set_title(title, fontsize=11.8, fontweight='bold', color=ENDPOINT_COLORS[task], pad=9)
    ax.set_xlabel('Cumulative local SHAP contribution\n(background-centered)', fontsize=9.7)
    ax.tick_params(axis='x', labelsize=6.2)
    ax.tick_params(axis='y', length=0)
    clean_axes(ax, grid_axis='x')

def plot_representative_local_patient(grp):
    row = representative_patients[representative_patients['MBP_raw'] == grp]
    if row.empty:
        raise ValueError(f'No representative patient available for {grp}.')
    patient = str(row['sample'].iloc[0])
    panel = LOCAL_PANEL[grp]
    fig, axes = plt.subplots(1, 3, figsize=(13.4, 6.4), facecolor='white', sharey=False)
    source_frames = []
    for ax, task in zip(axes, ENDPOINT_ORDER):
        d = _get_local_contribution_table(patient=patient, task=task, top_n=TOP_LOCAL_FEATURES)
        d['sample'] = patient
        d['MBP_raw'] = grp
        d['outcome_clean'] = task
        source_frames.append(d)
        _plot_local_waterfall_axis(ax=ax, d=d, patient=patient, task=task)
    comp_rank = float(patient_order_df.loc[patient, 'composite_benefit_rank']) if patient in patient_order_df.index and 'composite_benefit_rank' in patient_order_df.columns else np.nan
    distance = float(row['robust_distance_to_group_median'].iloc[0])
    add_panel_header(fig, panel, f'Representative local MRU attribution profile — {grp}')
    fig.text(0.5, 0.925, f'Patient {patient} | distance to group median SHAP profile={distance:.2f}' + (f' | composite benefit rank={comp_rank:.2f}' if np.isfinite(comp_rank) else ''), ha='center', va='center', fontsize=10.0, color=COLOR_TEXT)
    source_df = pd.concat(source_frames, axis=0, ignore_index=True)
    source_df.to_csv(LOCAL_SOURCE_DIR / f"{panel}_{grp}{get_value('science.tables.attribution_040', 'attribution_040.csv')}", index=False, encoding='utf-8-sig')
    fig.subplots_adjust(left=0.14, right=0.98, top=0.84, bottom=0.08, wspace=0.46)
    save_figure(fig, LOCAL_OUT_DIR / f'{panel}_{grp}_representative_local_SHAP_waterfall_Nature_style')
    plt.show()
    return source_df
sxs_mbp1 = plot_representative_local_patient('MBP-I')
sxt_mbp2 = plot_representative_local_patient('MBP-II')
sxu_mbp3 = plot_representative_local_patient('MBP-III')





In [ ]:
SUPP_TABLE_DIR = OUT_DIR / get_value('science.paths.directory_025', 'directory_025')
SUPP_TABLE_SOURCE_DIR = SOURCE_OUT_DIR / get_value('science.paths.directory_025', 'directory_025')
for d in [SUPP_TABLE_DIR, SUPP_TABLE_SOURCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def _safe_merge_selected_outcomes():
    cols = ['modal_clean', 'feature_clean']
    if 'pattern' in golden.columns:
        cols.append('pattern')
    if 'outcome_list' in golden.columns:
        cols.append('outcome_list')
    d = golden[cols].drop_duplicates(['modal_clean', 'feature_clean']).copy()
    if 'outcome_list' in d.columns:
        d['selected_outcomes'] = d['outcome_list'].map(lambda x: '; '.join(map(str, x)) if isinstance(x, (list, tuple, set)) else str(x))
        d = d.drop(columns=['outcome_list'])
    elif 'pattern' in d.columns:
        d['selected_outcomes'] = d['pattern'].astype(str)
    else:
        d['selected_outcomes'] = ''
    return d

def _endpoint_wide(df, value_cols, prefix_map=None):
    prefix_map = prefix_map or {'Response': 'Response', 'PFS': 'PFS', 'OS': 'OS'}
    pieces = []
    base = df[['modal_clean', 'feature_clean']].drop_duplicates().copy()
    for task in ENDPOINT_ORDER:
        ss = df[df['outcome_clean'] == task].copy()
        if ss.empty:
            continue
        keep = ['modal_clean', 'feature_clean'] + [c for c in value_cols if c in ss.columns]
        ss = ss[keep].copy()
        rename_map = {c: f'{prefix_map.get(task, task)}_{c}' for c in value_cols if c in ss.columns}
        ss = ss.rename(columns=rename_map)
        pieces.append(ss)
    out = base
    for ss in pieces:
        out = out.merge(ss, on=['modal_clean', 'feature_clean'], how='left')
    return out

def build_table_sx1():
    if 'stage1_stability_endpoint' not in globals():
        raise RuntimeError('Run the Figure 4F bootstrap-stability section before Table Sx1.')
    selected_meta = _safe_merge_selected_outcomes()
    d = stage1_stability_endpoint.copy()
    sx1 = _endpoint_wide(d, value_cols=['originally_selected', 'top_k', 'bootstrap_topk_frequency', 'bootstrap_median_rank', 'bootstrap_rank_q25', 'bootstrap_rank_q75', 'n_bootstrap'])
    sx1 = selected_meta.merge(sx1, on=['modal_clean', 'feature_clean'], how='left')
    if 'stage1_stability_feature' in globals():
        agg_cols = ['modal_clean', 'feature_clean', 'mean_topk_frequency', 'min_topk_frequency', 'median_bootstrap_rank', 'n_selected_endpoints']
        available = [c for c in agg_cols if c in stage1_stability_feature.columns]
        sx1 = sx1.merge(stage1_stability_feature[available].drop_duplicates(['modal_clean', 'feature_clean']), on=['modal_clean', 'feature_clean'], how='left')
    modality_rank = {m: i for i, m in enumerate(MODALITY_ORDER_STAGE1)}
    sx1['_modal_rank'] = sx1['modal_clean'].map(modality_rank).fillna(99)
    sort_cols = ['_modal_rank']
    if 'n_selected_endpoints' in sx1.columns:
        sort_cols.append('n_selected_endpoints')
    if 'mean_topk_frequency' in sx1.columns:
        sort_cols.append('mean_topk_frequency')
    sort_cols.append('feature_clean')
    ascending = [True, *([False] if 'n_selected_endpoints' in sort_cols else []), *([False] if 'mean_topk_frequency' in sort_cols else []), True]
    sx1 = sx1.sort_values(sort_cols, ascending=ascending).drop(columns=['_modal_rank']).reset_index(drop=True)
    rename = {'modal_clean': 'Modality', 'feature_clean': 'Feature', 'pattern': 'Original selection pattern', 'selected_outcomes': 'Originally selected endpoint(s)', 'mean_topk_frequency': 'Mean bootstrap Top-k retention frequency', 'min_topk_frequency': 'Minimum bootstrap Top-k retention frequency', 'median_bootstrap_rank': 'Median bootstrap rank across selected endpoints', 'n_selected_endpoints': 'Number of originally selected endpoints'}
    for task in ENDPOINT_ORDER:
        rename.update({f'{task}_originally_selected': f'{task}: originally selected', f'{task}_top_k': f'{task}: original Top-k', f'{task}_bootstrap_topk_frequency': f'{task}: bootstrap Top-k retention frequency', f'{task}_bootstrap_median_rank': f'{task}: bootstrap median rank', f'{task}_bootstrap_rank_q25': f'{task}: bootstrap rank Q1', f'{task}_bootstrap_rank_q75': f'{task}: bootstrap rank Q3', f'{task}_n_bootstrap': f'{task}: bootstrap resamples'})
    sx1 = sx1.rename(columns=rename)
    preferred = ['Modality', 'Feature', 'Originally selected endpoint(s)', 'Original selection pattern', 'Number of originally selected endpoints', 'Mean bootstrap Top-k retention frequency', 'Minimum bootstrap Top-k retention frequency', 'Median bootstrap rank across selected endpoints']
    endpoint_cols = []
    for task in ENDPOINT_ORDER:
        endpoint_cols.extend([f'{task}: originally selected', f'{task}: original Top-k', f'{task}: bootstrap Top-k retention frequency', f'{task}: bootstrap median rank', f'{task}: bootstrap rank Q1', f'{task}: bootstrap rank Q3', f'{task}: bootstrap resamples'])
    ordered = [c for c in preferred + endpoint_cols if c in sx1.columns]
    other = [c for c in sx1.columns if c not in ordered]
    return sx1[ordered + other]
table_sx1 = build_table_sx1()

def build_table_sx2():
    if 'feature_preference' not in globals() or 'determinant_long' not in globals():
        raise RuntimeError('Run the Figure 6 feature-preference section before Table Sx2.')
    pref = feature_preference.copy()
    imp = determinant_long[determinant_long['outcome_clean'].isin(ENDPOINT_ORDER)].copy()
    imp['endpoint_rank'] = imp.groupby('outcome_clean')['mean_abs_shap_clean'].rank(method='min', ascending=False)
    imp_wide = _endpoint_wide(imp, value_cols=['mean_abs_shap_clean', 'direction_rho', 'endpoint_rank'])
    sx2 = pref.merge(imp_wide, on=['modal_clean', 'feature_clean'], how='left')
    rename = {'modal_clean': 'Modality', 'feature_clean': 'Feature', 'overall_relative_importance': 'Overall relative importance', 'sharedness_entropy': 'Sharedness entropy', 'shared_score': 'Shared score', 'preference_margin': 'Endpoint preference margin', 'dominant_endpoint': 'Dominant endpoint'}
    for task in ENDPOINT_ORDER:
        rename.update({task: f'{task}: relative |SHAP| importance', f'p_{task}': f'{task}: across-endpoint attribution share', f'{task}_preference_score': f'{task}: preference score', f'{task}_mean_abs_shap_clean': f'{task}: mean |SHAP|', f'{task}_direction_rho': f'{task}: feature–SHAP Spearman rho', f'{task}_endpoint_rank': f'{task}: importance rank'})
    sx2 = sx2.rename(columns=rename)
    ordered = ['Modality', 'Feature', 'Overall relative importance', 'Sharedness entropy', 'Shared score', 'Dominant endpoint', 'Endpoint preference margin']
    for task in ENDPOINT_ORDER:
        ordered.extend([f'{task}: mean |SHAP|', f'{task}: relative |SHAP| importance', f'{task}: importance rank', f'{task}: across-endpoint attribution share', f'{task}: preference score', f'{task}: feature–SHAP Spearman rho'])
    ordered = [c for c in ordered if c in sx2.columns]
    other = [c for c in sx2.columns if c not in ordered]
    sx2 = sx2[ordered + other]
    return sx2.sort_values('Overall relative importance', ascending=False).reset_index(drop=True)
table_sx2 = build_table_sx2()

def build_table_sx3():
    if 'sxf_long' not in globals():
        raise RuntimeError('Run the SxF alternative-model sensitivity section before Table Sx3.')
    d = sxf_long.copy()
    support_all = d.groupby(['feature_clean', 'modal_clean'], as_index=False).agg(Alternative_model_task_count=('topk_support', 'count'), Alternative_Top10_support_count=('topk_support', 'sum'), Alternative_mean_normalized_importance=('importance_norm', 'mean'), Alternative_median_rank=('rank', 'median'))
    support_all['Alternative_Top10_support_fraction'] = support_all['Alternative_Top10_support_count'] / support_all['Alternative_model_task_count'].replace(0, np.nan)
    dnn = determinant_long[determinant_long['outcome_clean'].isin(ENDPOINT_ORDER)].groupby(['feature_clean', 'modal_clean'], as_index=False).agg(DNN_mean_abs_SHAP=('mean_abs_shap_clean', 'mean'))
    dnn['DNN_overall_normalized_importance'] = dnn['DNN_mean_abs_SHAP'] / dnn['DNN_mean_abs_SHAP'].sum()
    sx3 = d.merge(support_all, on=['feature_clean', 'modal_clean'], how='left').merge(dnn, on=['feature_clean', 'modal_clean'], how='left')
    sx3 = sx3.rename(columns={'task': 'Endpoint', 'model_family': 'Alternative model family', 'feature_clean': 'Feature', 'modal_clean': 'Modality', 'importance_norm': 'Alternative-model normalized importance', 'rank': 'Alternative-model importance rank', 'topk_support': 'Top-10 supported', 'Alternative_model_task_count': 'Number of available model–endpoint combinations', 'Alternative_Top10_support_count': 'Top-10 support count across model–endpoint combinations', 'Alternative_mean_normalized_importance': 'Mean alternative-model normalized importance', 'Alternative_median_rank': 'Median alternative-model importance rank', 'Alternative_Top10_support_fraction': 'Top-10 support fraction across model–endpoint combinations', 'DNN_mean_abs_SHAP': 'DNN mean |SHAP| across endpoints', 'DNN_overall_normalized_importance': 'DNN overall normalized importance'})
    endpoint_rank = {t: i for i, t in enumerate(ENDPOINT_ORDER)}
    sx3['_endpoint_rank'] = sx3['Endpoint'].map(endpoint_rank).fillna(99)
    sx3 = sx3.sort_values(['_endpoint_rank', 'Alternative model family', 'Alternative-model importance rank', 'Feature']).drop(columns=['_endpoint_rank']).reset_index(drop=True)
    ordered = ['Endpoint', 'Alternative model family', 'Modality', 'Feature', 'Alternative-model normalized importance', 'Alternative-model importance rank', 'Top-10 supported', 'Number of available model–endpoint combinations', 'Top-10 support count across model–endpoint combinations', 'Top-10 support fraction across model–endpoint combinations', 'Mean alternative-model normalized importance', 'Median alternative-model importance rank', 'DNN mean |SHAP| across endpoints', 'DNN overall normalized importance']
    return sx3[[c for c in ordered if c in sx3.columns] + [c for c in sx3.columns if c not in ordered]]
table_sx3 = build_table_sx3()

def build_table_sx4():
    if 'sxr_stats' not in globals():
        raise RuntimeError('Run the SxR extended concordance section before Table Sx4.')
    sx4 = sxr_stats.copy()
    sx4 = sx4.rename(columns={'strata_type': 'Stratification type', 'stratum': 'Stratum', 'pair': 'Endpoint pair', 'n': 'Patients, n', 'median_rho': 'Median Spearman rho', 'q25': 'Spearman rho Q1', 'q75': 'Spearman rho Q3', 'mean_rho': 'Mean Spearman rho', 'sd_rho': 'SD of Spearman rho'})
    sx4['Spearman rho IQR'] = sx4['Spearman rho Q1'].map(lambda x: f'{x:.3f}' if pd.notna(x) else '') + '–' + sx4['Spearman rho Q3'].map(lambda x: f'{x:.3f}' if pd.notna(x) else '')
    strata_order = {'Overall': 0, 'MBP-I': 1, 'MBP-II': 2, 'MBP-III': 3, 'Radiomics': 4, 'Metagenomics': 5, 'Urine': 6}
    pair_order = {'Response–PFS': 0, 'Response–OS': 1, 'PFS–OS': 2}
    sx4['_stratum_rank'] = sx4['Stratum'].map(strata_order).fillna(99)
    sx4['_pair_rank'] = sx4['Endpoint pair'].map(pair_order).fillna(99)
    sx4 = sx4.sort_values(['_stratum_rank', '_pair_rank']).drop(columns=['_stratum_rank', '_pair_rank']).reset_index(drop=True)
    ordered = ['Stratification type', 'Stratum', 'Endpoint pair', 'Patients, n', 'Median Spearman rho', 'Spearman rho IQR', 'Spearman rho Q1', 'Spearman rho Q3', 'Mean Spearman rho', 'SD of Spearman rho']
    return sx4[[c for c in ordered if c in sx4.columns] + [c for c in sx4.columns if c not in ordered]]
table_sx4 = build_table_sx4()
TABLE_SPECS = [('Table_Sx1', 'Table Sx1. Stage-1 SHAP feature prioritization and bootstrap robustness', table_sx1, 'Stage-1 retained features, original endpoint selection, and patient-resampling bootstrap ranking robustness.'), ('Table_Sx2', 'Table Sx2. Complete attribution characteristics of the final MRU determinants', table_sx2, 'Complete final-MRU determinant ledger: endpoint attribution magnitude/rank, sharedness, and endpoint-preference metrics.'), ('Table_Sx3', 'Table Sx3. Alternative-model sensitivity of final MRU determinants', table_sx3, 'Feature importance and Top-10 support across the available alternative model–endpoint combinations.'), ('Table_Sx4', 'Table Sx4. Cross-endpoint concordance of patient-level MRU attribution profiles', table_sx4, 'Descriptive cross-endpoint SHAP-profile concordance overall and after MBP/modality stratification.')]
csv_paths = {}
for sheet_name, title, df, note in TABLE_SPECS:
    csv_path = SUPP_TABLE_SOURCE_DIR / f'{sheet_name}.csv'
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    csv_paths[sheet_name] = csv_path
    

table_manifest = pd.DataFrame([{'Table': sheet_name, 'Title': title, 'Rows': int(df.shape[0]), 'Columns': int(df.shape[1]), 'CSV': str(csv_paths[sheet_name])} for sheet_name, title, df, note in TABLE_SPECS])
table_manifest.to_csv(SUPP_TABLE_SOURCE_DIR / get_value('science.tables.result_224', 'result_224.csv'), index=False, encoding='utf-8-sig')




